# Side-Channel Attacks Survive Noise Cancellation in 3D Printers

**Dataset:** Madamopoulos & Tsoutsos (2024), *3D Printer Audio and Vibration Side
Channel Dataset*, Zenodo [10.5281/zenodo.13329934](https://doi.org/10.5281/zenodo.13329934)
(described in *Data in Brief* 57, 111002). 144 synchronised audio–vibration pairs,
12 object classes, Bambu Lab P1P (core-XY) and A1 Mini (bed-slinger).

**Research question:** does an active acoustic suppression mechanism eliminate the
channel it attenuates, and what dominates leakage once confounds are controlled?

---

## How this notebook is organised

Each stage below is **standalone** — it rebuilds its own catalog, extracts its own
features, and writes structured JSON results. Stages can be run independently and in
any order. Every reported value comes from one of these stages.

| Stage | Produces |
|---|---|
| 1 · Acoustic | observation-window sweep, truncation audit, notch sweep, permutation tests |
| 2 · Main analysis | duration, vibration, per-configuration, per-printer, transfer, temporal |
| 3 · G-code ground truth | recording duration vs sliced print time |
| 4 · Spectral + exploratory | motor-band suppression vs background |
| 5 · Duration controls | T1 incremental, fixed-offset window, T2 truncation, T3 subset |
| 6 · Figures | the five summary figures |

## Reproducibility

Deterministic catalog ordering, fixed random seeds, and fixed cuDNN algorithm
selection throughout. Each stage verifies catalog composition (144 recordings,
12 classes, 2 printers) before running and aborts on mismatch. Permutation tests use
B = 1000 with the (r+1)/(B+1) estimator.

## Note on the two capture configurations

The dataset mixes two capture rigs, balanced six and six per class: a Teensy 4.0 with a
mounted accelerometer (measured 500.0 Hz) and an iPhone application (measured 199.6 Hz).
Vibration features are therefore computed in physical units, and observation windows are
equalised **in seconds** rather than in samples.

## Stored outputs

Outputs are preserved so the notebook is self-verifying: the printed values can be read
without re-running. Re-running reproduces them.


## Setup

Mount Drive and fetch the dataset from Zenodo. Skip the download if the dataset is
already present.


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/3dprinter_sidechannel'
OUTPUT_DIR = '/content/drive/MyDrive/3dprinter_sidechannel/results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Dataset base: {DRIVE_BASE}')
print(f'Results will be saved to: {OUTPUT_DIR}')

Mounted at /content/drive
Dataset base: /content/drive/MyDrive/3dprinter_sidechannel
Results will be saved to: /content/drive/MyDrive/3dprinter_sidechannel/results


In [ ]:
# ── Auto-download Zenodo dataset to Google Drive ──────────────────────────────
!pip install -q zenodo-get

import os
DRIVE_BASE = '/content/drive/MyDrive/3dprinter_sidechannel'
os.makedirs(DRIVE_BASE, exist_ok=True)

# Download directly from Zenodo into the Drive folder
os.chdir(DRIVE_BASE)
!zenodo_get 13329934

# Show what was downloaded
for root, dirs, files in os.walk(DRIVE_BASE):
    depth = root.replace(DRIVE_BASE, '').count(os.sep)
    if depth > 3: continue
    print('  ' * depth + os.path.basename(root) + '/')
    for f in files[:5]:
        print('  ' * (depth+1) + f)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.6/254.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 7.1 MB/s eta 0:00:00
INFO: Output directory: /content/drive/MyDrive/3dprinter_sidechannel
INFO: Title: 3D printer audio and vibration side channels
INFO: Total size: 10.8 GB
INFO: Number of files: 16
INFO: 05_2keys.zip is already downloaded correctly.
INFO: 07_NIST_additive_test.zip is already downloaded correctly.
INFO: 08_ASTM.zip is already downloaded correctly.
INFO: 10_Triple_Helix.zip is already downloaded correctly.
INFO: Printing Videos.zip is already downloaded correctly.
INFO: 12_retraction_test.zip is already downloaded correctly.
INFO: 11_CaliCat.zip is already downloaded correctly.
INFO: Noise Recordings.zip is already downloaded correctly.
INFO: 04_key_steps.zip is already downloaded

---
## Stage 1 · Acoustic channel

Observation-window sweep with truncation audit, simulated notch-bank sweep, and
label-permutation tests. Produces the observation-window results and the values behind the first summary figure.

Key results: 11.11% at 30 s (permutation *p* = 0.188), **27.08% at a duration-clean
60 s window**, 40.28% under distributed sampling, against an 8.33% baseline.


In [ ]:
# =============================================================================
#  STAGE 1 — ACOUSTIC CHANNEL
#
#  Observation-window sweep (10/30/60/120/300 s + distributed sampling), truncation
#  audit, simulated notch-bank quality-factor sweep, and label-permutation tests
#  (B = 1000). Standalone: rebuilds its own catalog and features.
# =============================================================================

!pip install -q librosa scikit-learn statsmodels pandas numpy matplotlib scipy

import os, json, time, warnings, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import librosa
from scipy.signal import iirnotch, filtfilt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import (StratifiedKFold, StratifiedGroupKFold,
                                     cross_val_predict)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.feature_selection import mutual_info_classif
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped:", e)

# ======================= CONFIG — EDIT THESE IF NEEDED =======================
DRIVE_BASE  = '/content/drive/MyDrive/3dprinter_sidechannel'   # dataset root
OUT         = '/content/drive/MyDrive/3dprinter_sidechannel/revision_v3'

# Optional: a NON-AMNC dataset laid out the same way (e.g. Costa et al.
# Ultimaker 3). Leave as None to skip. This is the closest thing to an
# AMNC-off control that the data permit, and it is what R1-1 is really asking
# for. Highly recommended if you can get it.
EXTERNAL_NONAMNC_DIR = None

B_PERM        = 1000          # permutations. 200 floors p at 0.005 — R1 noticed.
SEEDS         = [0, 1, 2]     # multi-seed for the temporal model
ACOUSTIC_CAP  = 30            # seconds — the value under review
WINDOW_SWEEP  = [10, 30, 60, 120, 300]   # A2: is 30 s the reason acoustic fails?
N_MULTIWIN    = 8             # A2: 30 s windows spread across the whole print
NOTCH_QS      = [15, 30, 60]
T_PROP        = 120           # original proportional segmentation
SEG_SAMPLES   = 2048          # A7b: fixed-length segment (duration-matched)
RANDOM_STATE  = 42
RUN_TEMPORAL  = True          # needs torch; set False for a fast summary-only run
# =============================================================================

os.makedirs(OUT, exist_ok=True)
np.random.seed(RANDOM_STATE)
RESULTS = {}
t_start = time.time()

plt.rcParams.update({'font.size': 12, 'font.family': 'serif',
                     'figure.dpi': 300, 'savefig.dpi': 300,
                     'savefig.bbox': 'tight', 'axes.grid': True,
                     'grid.alpha': 0.3, 'grid.linestyle': '--'})

BASE = Path(DRIVE_BASE)   # Drive only: never let a stale local copy shadow it
SKIP = {'results', 'results_v2', 'revision_v3', 'Printing Videos',
        'Noise Recordings', 'Artifacts', '__MACOSX'}
AUD  = {'.mp3', '.caf', '.wav', '.m4a'}
print(f"Dataset root : {BASE}")
print(f"Output root  : {OUT}\n")


def wilson(acc_pct, n):
    """Wilson 95% CI for an accuracy given as a percentage."""
    if n == 0:
        return (float('nan'), float('nan'))
    lo, hi = proportion_confint(int(round(acc_pct / 100 * n)), n, 0.05, 'wilson')
    return round(lo * 100, 2), round(hi * 100, 2)


# ----------------------------------------------------------------------------
# A0. DATASET METADATA SCRAPE  ->  sensor placement / physical threat model
#     R1-4 and R2 both want accelerometer placement + physical access detail.
#     This pulls every README / metadata / doc file so you can quote the source
#     rather than guess. Anything it cannot find must be stated as unknown.
# ----------------------------------------------------------------------------
print("=" * 78)
print("A0  DATASET METADATA (sensor placement evidence)")
print("=" * 78)
meta_files, meta_text = [], []
for p in BASE.rglob('*'):
    if p.is_file() and p.suffix.lower() in {'.txt', '.md', '.json', '.pdf', '.rst'}:
        if any(s in p.parts for s in SKIP):
            continue
        meta_files.append(str(p.relative_to(BASE)))
        if p.suffix.lower() in {'.txt', '.md', '.rst', '.json'}:
            try:
                meta_text.append(f"\n### {p.relative_to(BASE)}\n"
                                 + p.read_text(errors='ignore')[:4000])
            except Exception:
                pass
print(f"metadata/doc files found: {len(meta_files)}")
for m in meta_files[:40]:
    print("   ", m)
Path(f"{OUT}/dataset_metadata_dump.md").write_text(
    "# Dataset metadata dump (for threat-model section)\n"
    "Search this for: accelerometer mount point, microphone/iPhone position and\n"
    "distance, sampling rates, axis orientation, re-mounting between runs.\n"
    + "\n".join(meta_text))
RESULTS['metadata_files'] = meta_files
print(f"\n-> {OUT}/dataset_metadata_dump.md")
print("   Read it. Whatever it does not specify, declare as UNKNOWN in the paper.\n")


# ----------------------------------------------------------------------------
# 1. PAIRING CATALOG — one source of truth for every number below
# ----------------------------------------------------------------------------
def build_catalog(root):
    root, rows = Path(root), []
    for obj in sorted(root.iterdir()):
        if not obj.is_dir() or obj.name in SKIP or obj.name.endswith('.zip'):
            continue
        nested = [d for d in obj.iterdir() if d.is_dir()]
        if not nested:
            continue
        for pr in sorted(nested[0].iterdir()):
            if not pr.is_dir():
                continue
            sess = {}
            for f in pr.rglob('*'):
                e = f.suffix.lower()
                if e in AUD or e == '.csv':
                    s = sess.setdefault(f.parent, {'a': [], 'c': []})
                    (s['a'] if e in AUD else s['c']).append(f)
            for folder, fs in sess.items():
                if fs['a'] and fs['c']:
                    for af in fs['a']:
                        rows.append({'label': obj.name, 'printer': pr.name,
                                     'session': str(folder), 'audio': str(af),
                                     'vibr': str(fs['c'][0])})
    return pd.DataFrame(rows)


cat = build_catalog(BASE)
cat['group'] = LabelEncoder().fit_transform(cat['session'])
print("=" * 78)
print("1  CATALOG")
print("=" * 78)
print(f"paired recordings {len(cat)} | classes {cat['label'].nunique()} "
      f"| printers {cat['printer'].nunique()} | sessions {cat['group'].nunique()}")

# A10: is the "grouped CV" claim doing any work?
spg = cat.groupby('group').size()
grouping_informative = bool((spg > 1).any())
print(f"samples per session: min {spg.min()} max {spg.max()} mean {spg.mean():.2f}")
print("GROUPING IS INFORMATIVE" if grouping_informative else
      "GROUPING IS VACUOUS -- every session has one sample, so GroupKFold == KFold.\n"
      "   Do not claim grouped CV as a validity safeguard in the paper if this prints.")
RESULTS['n_samples']  = int(len(cat))
RESULTS['n_sessions'] = int(cat['group'].nunique())
RESULTS['samples_per_group_max'] = int(spg.max())
RESULTS['grouping_informative']  = grouping_informative


# ----------------------------------------------------------------------------
# 2. FEATURE EXTRACTION
# ----------------------------------------------------------------------------
def audio_duration(p):
    for kw in ('path', 'filename'):
        try:
            return float(librosa.get_duration(**{kw: p}))
        except Exception:
            continue
    return np.nan


def amnc_notch(y, sr, Q=30):
    for f0 in [120, 180, 240, 300, 360]:
        if f0 < sr / 2:
            b, a = iirnotch(f0, Q, sr)
            y = filtfilt(b, a, y)
    return y


def acoustic_vec(y, sr):
    m  = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    c  = librosa.feature.spectral_centroid(y=y, sr=sr)
    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    ro = librosa.feature.spectral_rolloff(y=y, sr=sr)
    z  = librosa.feature.zero_crossing_rate(y)
    return np.concatenate([m.mean(1), m.std(1), c.mean(1), c.std(1),
                           bw.mean(1), bw.std(1), ro.mean(1), z.mean(1)])


def acoustic_feats(p, dur=ACOUSTIC_CAP, offset=0.0, defend=False, Q=30):
    try:
        y, sr = librosa.load(p, sr=16000, mono=True, offset=offset, duration=dur)
    except Exception:
        return None
    if len(y) < 1024:
        return None
    if defend:
        y = amnc_notch(y, sr, Q)
    return acoustic_vec(y, sr)


def load_xyz(p):
    try:
        df = pd.read_csv(p)
    except Exception:
        return None, None
    df.columns = [str(c).strip().lower() for c in df.columns]
    if not {'x', 'y', 'z'} <= set(df.columns):
        return None, None
    a = df[['x', 'y', 'z']].apply(pd.to_numeric, errors='coerce').dropna().to_numpy(float)
    # recover sample rate if a time column exists (needed to talk about seconds)
    fs = None
    for tc in ('t', 'time', 'timestamp', 'millis', 'micros', 'ms', 'us'):
        if tc in df.columns:
            tv = pd.to_numeric(df[tc], errors='coerce').dropna().to_numpy(float)
            if len(tv) > 10 and tv[-1] > tv[0]:
                span = tv[-1] - tv[0]
                if tc in ('millis', 'ms'):
                    span /= 1e3
                elif tc in ('micros', 'us'):
                    span /= 1e6
                fs = (len(tv) - 1) / span
            break
    return (a if len(a) >= 10 else None), fs


def vib_summary(a):
    out = []
    for ax in range(3):
        v = a[:, ax]
        mag = np.abs(np.fft.rfft(v))
        out += [v.mean(), v.std(), np.sqrt((v ** 2).mean()),
                v.max() - v.min(), np.fft.rfftfreq(len(v))[mag.argmax()]]
    return np.array(out)


def seq_proportional(a, T=T_PROP):
    """ORIGINAL segmentation: T segments regardless of length -> segment
    duration scales with recording duration. This is the duration confound."""
    b, rows = np.linspace(0, len(a), T + 1).astype(int), []
    for i in range(T):
        seg = a[b[i]:b[i + 1]]
        if len(seg) < 4:
            seg = a[b[i]:b[i] + 4]
        r = []
        for ax in range(3):
            v = seg[:, ax]
            mg = np.abs(np.fft.rfft(v - v.mean()))
            dom = np.fft.rfftfreq(len(v))[mg.argmax()] if len(v) > 1 else 0.0
            r += [v.std(), np.sqrt((v ** 2).mean()), v.max() - v.min(), dom]
        rows.append(r)
    return np.asarray(rows, np.float32)


def seq_fixed(a, n_seg, seg=SEG_SAMPLES):
    """DURATION-MATCHED segmentation: identical segment length AND identical
    total observed span for every recording (first n_seg*seg samples)."""
    rows = []
    for i in range(n_seg):
        s = a[i * seg:(i + 1) * seg]
        r = []
        for ax in range(3):
            v = s[:, ax]
            mg = np.abs(np.fft.rfft(v - v.mean()))
            dom = np.fft.rfftfreq(len(v))[mg.argmax()] if len(v) > 1 else 0.0
            r += [v.std(), np.sqrt((v ** 2).mean()), v.max() - v.min(), dom]
        rows.append(r)
    return np.asarray(rows, np.float32)


print("\n" + "=" * 78)
print("2  FEATURE EXTRACTION")
print("=" * 78)
rows, raw_vib = [], []
for i, (_, r) in enumerate(cat.iterrows()):
    if i % 20 == 0:
        print(f"   {i}/{len(cat)}", flush=True)
    a_raw, fs = load_xyz(r['vibr'])
    if a_raw is None:
        continue
    ac = acoustic_feats(r['audio'])
    if ac is None:
        continue
    rows.append({'label': r['label'], 'printer': r['printer'],
                 'group': r['group'], 'audio': r['audio'], 'vibr': r['vibr'],
                 'a': ac,
                 'ad': acoustic_feats(r['audio'], defend=True),
                 'v': vib_summary(a_raw),
                 'audio_dur': audio_duration(r['audio']),
                 'vib_n': len(a_raw), 'vib_fs': fs})
    raw_vib.append(a_raw)

R  = pd.DataFrame(rows)
Xa = np.nan_to_num(np.vstack(R['a']))
Xd = np.nan_to_num(np.vstack(R['ad']))
Xv = np.nan_to_num(np.vstack(R['v']))
le = LabelEncoder().fit(R['label'])
y  = le.transform(R['label'])
g  = R['group'].to_numpy()
pr = R['printer'].to_numpy()
ncls   = len(le.classes_)
chance = 100 / ncls
VN = [f'{ax}_{s}' for ax in ['X', 'Y', 'Z']
      for s in ['mean', 'std', 'rms', 'p2p', 'fft']]
print(f"kept {len(R)} samples | {ncls} classes | chance {chance:.2f}%")
RESULTS['chance'] = round(chance, 2)
RESULTS['n_kept'] = int(len(R))
RESULTS['vib_fs_detected'] = None if R['vib_fs'].isna().all() else float(R['vib_fs'].median())


# ----------------------------------------------------------------------------
# 3. EVALUATION HELPERS
# ----------------------------------------------------------------------------
def rf():
    return Pipeline([('s', StandardScaler()),
                     ('m', RandomForestClassifier(n_estimators=200,
                                                  random_state=RANDOM_STATE,
                                                  n_jobs=-1))])


def cv_acc(X, yy, gg=None, clf=None, folds=5):
    """Grouped CV when groups are informative, stratified CV otherwise."""
    X, clf = np.nan_to_num(X), (clf or rf())
    if gg is not None and len(np.unique(gg)) < len(yy):
        splitter = StratifiedGroupKFold(folds, shuffle=True,
                                        random_state=RANDOM_STATE)
        yp = cross_val_predict(clf, X, yy, groups=gg, cv=splitter)
    else:
        splitter = StratifiedKFold(folds, shuffle=True,
                                   random_state=RANDOM_STATE)
        yp = cross_val_predict(clf, X, yy, cv=splitter)
    return (round(accuracy_score(yy, yp) * 100, 2),
            round(f1_score(yy, yp, average='macro') * 100, 2), yp)


def permutation_p(X, yy, gg, observed, B=B_PERM, tag=""):
    """Label-permutation null. Labels are permuted WITHIN the design; the
    p-value estimator is (r+1)/(B+1), which is bounded below by 1/(B+1).
    Report B in the paper -- R1 asked for exactly this."""
    rng, r = np.random.default_rng(RANDOM_STATE), 0
    for b in range(B):
        if b % max(1, B // 10) == 0:
            print(f"      perm {tag} {b}/{B}", flush=True)
        if cv_acc(X, rng.permutation(yy), gg)[0] >= observed:
            r += 1
    return round((r + 1) / (B + 1), 5)


# ----------------------------------------------------------------------------
# A1. ACOUSTIC CHANNEL UNDER AMNC  (+ permutation null, which the paper lacks)
# ----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("A1  ACOUSTIC UNDER AMNC")
print("=" * 78)
ac  = cv_acc(Xa, y, g)
acd = cv_acc(Xd, y, g)
RESULTS['acoustic_amnc']        = ac[0]
RESULTS['acoustic_amnc_ci']     = wilson(ac[0], len(y))
RESULTS['acoustic_amnc_f1']     = ac[1]
RESULTS['acoustic_simnotch']    = acd[0]
print(f"acoustic (AMNC active) {ac[0]:.2f}%  CI {RESULTS['acoustic_amnc_ci']}")
print(f"acoustic (+sim notch)  {acd[0]:.2f}%")
RESULTS['acoustic_perm_p'] = permutation_p(Xa, y, g, ac[0], tag="acoustic")
print(f"permutation p = {RESULTS['acoustic_perm_p']}  (B={B_PERM})")

sens = {}
for Q in NOTCH_QS:
    XQ = np.nan_to_num(np.vstack([acoustic_feats(p, defend=True, Q=Q)
                                  for p in R['audio']]))
    sens[Q] = cv_acc(XQ, y, g)[0]
    print(f"   Q={Q}: {sens[Q]:.2f}%")
RESULTS['notch_sensitivity'] = sens


# ----------------------------------------------------------------------------
# A2. WHY 30 SECONDS?  — window-length sensitivity + multi-window voting
#     If the acoustic null holds at every window length AND under whole-print
#     sampling, the null is a property of the signal, not of the 30 s cap.
# ----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("A2  ACOUSTIC WINDOW-LENGTH SENSITIVITY  (answers R1's 30 s question)")
print("=" * 78)
win_sweep = {}
for W in WINDOW_SWEEP:
    XW = []
    ok = True
    for p in R['audio']:
        f = acoustic_feats(p, dur=W)
        if f is None:
            ok = False
            break
        XW.append(f)
    if not ok:
        continue
    win_sweep[W] = cv_acc(np.nan_to_num(np.vstack(XW)), y, g)[0]
    print(f"   {W:>4}s window -> {win_sweep[W]:.2f}%")
RESULTS['acoustic_window_sweep'] = win_sweep

# multi-window: N x 30 s spread over the ENTIRE print, features averaged
Xmw = []
for p, d in zip(R['audio'], R['audio_dur']):
    d = d if np.isfinite(d) else ACOUSTIC_CAP
    offs = np.linspace(0, max(0.0, d - ACOUSTIC_CAP), N_MULTIWIN)
    fs_ = [acoustic_feats(p, dur=ACOUSTIC_CAP, offset=float(o)) for o in offs]
    fs_ = [f for f in fs_ if f is not None]
    Xmw.append(np.mean(fs_, axis=0) if fs_ else np.zeros(Xa.shape[1]))
RESULTS['acoustic_multiwindow'] = cv_acc(np.nan_to_num(np.vstack(Xmw)), y, g)[0]
print(f"   {N_MULTIWIN} x {ACOUSTIC_CAP}s spread over whole print -> "
      f"{RESULTS['acoustic_multiwindow']:.2f}%")
print("   -> if all of these sit at chance, the 30 s cap is not the reason.")


# ----------------------------------------------------------------------------
# A3/A4. VIBRATION SUMMARY FEATURES
# ----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("A4  VIBRATION — SUMMARY FEATURES")
print("=" * 78)
vib = cv_acc(Xv, y, g)
RESULTS['vib_pooled']    = vib[0]
RESULTS['vib_pooled_ci'] = wilson(vib[0], len(y))
RESULTS['vib_pooled_f1'] = vib[1]
print(f"pooled {vib[0]:.2f}%  CI {RESULTS['vib_pooled_ci']}  F1 {vib[1]:.2f}")
RESULTS['vib_perm_p'] = permutation_p(Xv, y, g, vib[0], tag="vibration")
print(f"permutation p = {RESULTS['vib_perm_p']}  (B={B_PERM})")

within = {}
for p in sorted(set(pr)):
    m = pr == p
    a_, f_, _ = cv_acc(Xv[m], y[m], g[m])
    within[p] = {'acc': a_, 'ci': wilson(a_, int(m.sum())), 'f1': f_, 'n': int(m.sum())}
    print(f"   within {p}: {a_:.2f}%  CI {within[p]['ci']}")
RESULTS['vib_within_printer'] = within


# ----------------------------------------------------------------------------
# A5. DURATION CONFOUND  (R1-2)
#     5a  duration-only classifier: how much of the leak is just "how long?"
#     5b  duration-matched summary features: truncate every recording to the
#         shortest common sample count, then recompute the same statistics.
# ----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("A5  DURATION CONFOUND")
print("=" * 78)
Xdur = np.c_[R['audio_dur'].fillna(0).to_numpy(), R['vib_n'].to_numpy()]
dur_only = cv_acc(Xdur, y, g)
RESULTS['duration_only_acc'] = dur_only[0]
RESULTS['duration_only_ci']  = wilson(dur_only[0], len(y))
RESULTS['duration_mutual_info'] = round(float(
    mutual_info_classif(Xdur, y, random_state=RANDOM_STATE).sum()), 4)
print(f"5a duration-ONLY classifier : {dur_only[0]:.2f}%  "
      f"CI {RESULTS['duration_only_ci']}  (vibration was {vib[0]:.2f}%)")
print(f"   mutual information(duration; class) = {RESULTS['duration_mutual_info']}")
print("   HIGH here = duration is a confound and MUST be disclosed and controlled.")

min_n = int(min(len(a) for a in raw_vib))
Xv_trunc = np.nan_to_num(np.vstack([vib_summary(a[:min_n]) for a in raw_vib]))
trunc = cv_acc(Xv_trunc, y, g)
RESULTS['vib_duration_matched']    = trunc[0]
RESULTS['vib_duration_matched_ci'] = wilson(trunc[0], len(y))
RESULTS['duration_matched_samples'] = min_n
print(f"5b duration-MATCHED vibration (first {min_n} samples of every "
      f"recording): {trunc[0]:.2f}%  CI {RESULTS['vib_duration_matched_ci']}")
print("   Survives -> the leak is not a length artefact. Collapses -> reframe.")

durstats = (R.assign(lab=R['label'])
              .groupby('lab')['audio_dur']
              .agg(['mean', 'min', 'max', 'count']).round(1))
durstats.to_csv(f"{OUT}/table_durations.csv")
print(f"   per-class duration table -> {OUT}/table_durations.csv")


# ----------------------------------------------------------------------------
# A6. FEATURE ABLATION — amplitude vs waveform shape
# ----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("A6  FEATURE ABLATION")
print("=" * 78)
amp_i  = [0, 1, 2, 3, 5, 6, 7, 8, 10, 11, 12, 13]
freq_i = [4, 9, 14]
stab_i = [1, 2, 6, 7, 11, 12]
RESULTS['vib_amplitude_only'] = cv_acc(Xv[:, amp_i], y, g)[0]
RESULTS['vib_frequency_only'] = cv_acc(Xv[:, freq_i], y, g)[0]
RESULTS['vib_lenstable_only'] = cv_acc(Xv[:, stab_i], y, g)[0]
print(f"amplitude(12) {RESULTS['vib_amplitude_only']:.2f}% | "
      f"frequency(3) {RESULTS['vib_frequency_only']:.2f}% | "
      f"length-stable std+rms(6) {RESULTS['vib_lenstable_only']:.2f}%")

fi = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE,
                            n_jobs=-1).fit(StandardScaler().fit_transform(Xv),
                                           y).feature_importances_
RESULTS['top_features'] = [(VN[i], round(float(fi[i]), 4))
                           for i in np.argsort(fi)[::-1][:5]]


# ----------------------------------------------------------------------------
# A7. TEMPORAL MODEL — proportional (original) vs duration-matched (new)
#     ordered vs order-shuffled, multi-seed. This is the headline claim, and
#     A7b is what makes it survive R1-2.
# ----------------------------------------------------------------------------
RESULTS['temporal'] = {}
if RUN_TEMPORAL:
    try:
        import torch, torch.nn as nn
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        print("\n" + "=" * 78)
        print(f"A7  TEMPORAL MODEL  (device={dev})")
        print("=" * 78)

        class SeqCNN(nn.Module):
            def __init__(s, n, c_in):
                super().__init__()
                s.net = nn.Sequential(
                    nn.Conv1d(c_in, 32, 5, padding=2, dilation=1),
                    nn.BatchNorm1d(32), nn.ReLU(),
                    nn.Conv1d(32, 32, 5, padding=4, dilation=2),
                    nn.BatchNorm1d(32), nn.ReLU(),
                    nn.Conv1d(32, 64, 5, padding=8, dilation=4),
                    nn.BatchNorm1d(64), nn.ReLU(),
                    nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                    nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.5),
                    nn.Linear(64, n))

            def forward(s, x):
                return s.net(x)

        def fit_predict(Xtr, ytr, Xte, seed, c_in):
            torch.manual_seed(seed)
            m = SeqCNN(ncls, c_in).to(dev)
            opt = torch.optim.Adam(m.parameters(), 1e-3, weight_decay=1e-3)
            lf = nn.CrossEntropyLoss()
            Xt = torch.tensor(Xtr).to(dev)
            yt = torch.tensor(ytr).long().to(dev)
            m.train()
            for _ in range(40):
                for idx in torch.randperm(len(Xt)).split(32):
                    if len(idx) < 2:
                        continue
                    opt.zero_grad()
                    lf(m(Xt[idx]), yt[idx]).backward()
                    opt.step()
            m.eval()
            with torch.no_grad():
                return m(torch.tensor(Xte).to(dev)).argmax(1).cpu().numpy()

        def run_seq(S, tag, shuffle):
            """S: (N, T, F) array. Returns mean/std accuracy over SEEDS."""
            accs = []
            c_in = S.shape[2]
            for seed in SEEDS:
                rng = np.random.default_rng(seed)
                X = []
                for s in S:
                    z = (s - s.mean(0)) / (s.std(0) + 1e-6)   # amplitude removed
                    if shuffle:
                        z = z[rng.permutation(len(z))]        # order destroyed
                    X.append(z.T.astype(np.float32))
                X = np.stack(X)
                pred = np.zeros(len(y), int)
                splitter = (StratifiedGroupKFold(5, shuffle=True, random_state=seed)
                            if len(np.unique(g)) < len(y) else
                            StratifiedKFold(5, shuffle=True, random_state=seed))
                sp = (splitter.split(X, y, g) if len(np.unique(g)) < len(y)
                      else splitter.split(X, y))
                for tr, te in sp:
                    pred[te] = fit_predict(X[tr], y[tr], X[te], seed, c_in)
                accs.append(accuracy_score(y, pred) * 100)
            m_, s_ = round(float(np.mean(accs)), 2), round(float(np.std(accs)), 2)
            print(f"   {tag:38s} {m_:5.2f}% +/- {s_:.2f}")
            return m_, s_

        # 7a original proportional segmentation (reproduces the submitted number)
        Sp = np.stack([seq_proportional(a) for a in raw_vib])
        o_m, o_s = run_seq(Sp, "proportional segments, ORDERED", False)
        s_m, s_s = run_seq(Sp, "proportional segments, SHUFFLED", True)
        RESULTS['temporal']['proportional_ordered']  = [o_m, o_s]
        RESULTS['temporal']['proportional_shuffled'] = [s_m, s_s]

        # 7b duration-matched segmentation — the control R1-2 implies
        n_seg = int(min(len(a) // SEG_SAMPLES for a in raw_vib))
        n_seg = max(8, min(n_seg, T_PROP))
        print(f"   duration-matched: {n_seg} segments x {SEG_SAMPLES} samples "
              f"(identical span for every recording)")
        Sf = np.stack([seq_fixed(a, n_seg) for a in raw_vib])
        fo_m, fo_s = run_seq(Sf, "FIXED-LENGTH segments, ORDERED", False)
        fs_m, fs_s = run_seq(Sf, "FIXED-LENGTH segments, SHUFFLED", True)
        RESULTS['temporal']['fixed_ordered']  = [fo_m, fo_s]
        RESULTS['temporal']['fixed_shuffled'] = [fs_m, fs_s]
        RESULTS['temporal']['n_fixed_segments'] = n_seg
        RESULTS['temporal']['seg_samples'] = SEG_SAMPLES
        print("\n   READ THIS: the FIXED-LENGTH ordered-vs-shuffled gap is the "
              "number\n   that survives the duration objection. Report it as "
              "primary if it holds.")
    except Exception as e:
        print("temporal block failed:", e)
        RESULTS['temporal']['error'] = str(e)


# ----------------------------------------------------------------------------
# A8. CROSS-PRINTER TRANSFER  (R2 generalisation)
# ----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("A8  CROSS-PRINTER TRANSFER")
print("=" * 78)
ps, cross = sorted(set(pr)), {}
for a_p in ps:
    for b_p in ps:
        if a_p == b_p:
            continue
        m = rf().fit(Xv[pr == a_p], y[pr == a_p])
        acc_ = accuracy_score(y[pr == b_p], m.predict(Xv[pr == b_p])) * 100
        cross[f"{a_p}->{b_p}"] = {'acc': round(acc_, 2),
                                  'ci': wilson(acc_, int((pr == b_p).sum()))}
        print(f"   {a_p} -> {b_p}: {acc_:.2f}%  CI {cross[f'{a_p}->{b_p}']['ci']}")
RESULTS['cross_printer'] = cross


# ----------------------------------------------------------------------------
# A9. CLASSIFIER ABLATION
# ----------------------------------------------------------------------------
clfs = {
    'Random Forest':     rf(),
    'Gradient Boosting': Pipeline([('s', StandardScaler()),
                                   ('m', GradientBoostingClassifier(
                                       n_estimators=100, random_state=RANDOM_STATE))]),
    'SVM (RBF)':         Pipeline([('s', StandardScaler()),
                                   ('m', SVC(kernel='rbf', C=10,
                                             random_state=RANDOM_STATE))]),
    'KNN (k=5)':         Pipeline([('s', StandardScaler()),
                                   ('m', KNeighborsClassifier(n_neighbors=5))]),
}
RESULTS['classifier_ablation'] = {k: cv_acc(Xv, y, g, clf=v)[0]
                                  for k, v in clfs.items()}
print("\nA9  classifier ablation:", RESULTS['classifier_ablation'])


# ----------------------------------------------------------------------------
# A12. OPTIONAL EXTERNAL NON-AMNC COMPARISON  (the real answer to R1-1)
# ----------------------------------------------------------------------------
if EXTERNAL_NONAMNC_DIR and Path(EXTERNAL_NONAMNC_DIR).exists():
    print("\n" + "=" * 78)
    print("A12  EXTERNAL NON-AMNC ACOUSTIC COMPARISON")
    print("=" * 78)
    ext = build_catalog(EXTERNAL_NONAMNC_DIR)
    Xe, ye = [], []
    for _, r in ext.iterrows():
        f = acoustic_feats(r['audio'])
        if f is not None:
            Xe.append(f)
            ye.append(r['label'])
    if len(Xe) > 20:
        ye = LabelEncoder().fit_transform(ye)
        e_acc = cv_acc(np.nan_to_num(np.vstack(Xe)), ye)[0]
        RESULTS['external_nonamnc_acoustic'] = {
            'acc': e_acc, 'n': len(ye), 'classes': int(len(set(ye))),
            'chance': round(100 / len(set(ye)), 2),
            'ci': wilson(e_acc, len(ye))}
        print(f"   non-AMNC acoustic {e_acc:.2f}% (chance "
              f"{100/len(set(ye)):.2f}%) vs AMNC {ac[0]:.2f}% "
              f"(chance {chance:.2f}%)")
        print("   -> quasi-experimental contrast. NOT causal. Say so explicitly.")
else:
    print("\nA12 skipped (EXTERNAL_NONAMNC_DIR not set). "
          "Strongly consider running it -- it is the closest available answer "
          "to R1's AMNC on/off objection.")


# ----------------------------------------------------------------------------
# 4. FIGURES — every value drawn from RESULTS. Nothing hardcoded.
# ----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("4  FIGURES")
print("=" * 78)


def save(fig, name):
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(f'{OUT}/{name}.{ext}')
    plt.close(fig)
    print("   saved", name)


def bars(ax, labels, vals, colors):
    b = ax.bar(labels, vals, color=colors, edgecolor='black')
    for bb, v in zip(b, vals):
        ax.text(bb.get_x() + bb.get_width() / 2, v + 1, f'{v:.1f}%',
                ha='center', fontsize=9)
    ax.axhline(chance, color='red', ls='--', label=f'Chance ({chance:.1f}%)')
    ax.set_ylabel('Accuracy (%)')
    ax.legend()
    return b


temporal_ord = RESULTS['temporal'].get('fixed_ordered',
                RESULTS['temporal'].get('proportional_ordered', [np.nan, 0]))
temporal_shf = RESULTS['temporal'].get('fixed_shuffled',
                RESULTS['temporal'].get('proportional_shuffled', [np.nan, 0]))

# fig1 — accuracy by method (temporal bar NO LONGER hardcoded)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
labs = ['Acoustic\n(AMNC)', 'Vibration\n(pooled)'] + \
       [f"Vib within\n{p.split('_')[-1]}" for p in within] + ['Full-seq\ntemporal']
vals = [ac[0], vib[0]] + [within[p]['acc'] for p in within] + [temporal_ord[0]]
bars(ax, labs, vals, ['#0072B2', '#009E73'] + ['#56B4E9'] * len(within) + ['#E69F00'])
ax.set_ylim(0, 100)
save(fig, 'fig1_accuracy')

# fig2 — amplitude vs shape
fig, ax = plt.subplots(figsize=(7, 4.2))
v2 = [vib[0], RESULTS['vib_amplitude_only'], RESULTS['vib_frequency_only'],
      RESULTS['vib_lenstable_only']]
bars(ax, ['All 15', 'Magnitude\n(12)', 'Frequency\n(3)', 'std+rms\n(6)'], v2,
     ['#999999', '#0072B2', '#CC79A7', '#56B4E9'])
ax.set_ylim(0, max(v2) + 12)
save(fig, 'fig2_amplitude_vs_shape')

# fig3 — temporal order (both segmentations side by side)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
grp = ['Proportional\nordered', 'Proportional\nshuffled',
       'Fixed-length\nordered', 'Fixed-length\nshuffled']
gv = [RESULTS['temporal'].get('proportional_ordered', [np.nan, 0])[0],
      RESULTS['temporal'].get('proportional_shuffled', [np.nan, 0])[0],
      temporal_ord[0], temporal_shf[0]]
ge = [RESULTS['temporal'].get('proportional_ordered', [np.nan, 0])[1],
      RESULTS['temporal'].get('proportional_shuffled', [np.nan, 0])[1],
      temporal_ord[1], temporal_shf[1]]
b = ax.bar(grp, gv, yerr=ge, capsize=5,
           color=['#E69F00', '#999999', '#D55E00', '#BBBBBB'], edgecolor='black')
for bb, v in zip(b, gv):
    if np.isfinite(v):
        ax.text(bb.get_x() + bb.get_width() / 2, v + 3, f'{v:.1f}%',
                ha='center', fontsize=9)
ax.axhline(chance, color='red', ls='--', label=f'Chance ({chance:.1f}%)')
ax.set_ylabel('Accuracy (%)')
ax.legend()
save(fig, 'fig3_temporal_order')

# fig4 — feature importance
fig, ax = plt.subplots(figsize=(7, 4.5))
idx = np.argsort(fi)[::-1][:10][::-1]
ax.barh([VN[i] for i in idx], fi[idx], color='#009E73', edgecolor='black')
ax.set_xlabel('Mean decrease in impurity')
save(fig, 'fig4_feature_importance')

# fig5 — confusion
cm = confusion_matrix(y, vib[2])
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
cl = [c.replace('_', ' ') for c in le.classes_]
ax.set_xticks(range(ncls)); ax.set_yticks(range(ncls))
ax.set_xticklabels(cl, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(cl, fontsize=8)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.grid(False)
fig.colorbar(im, fraction=0.046, pad=0.04)
save(fig, 'fig5_confusion')

# fig6 — cross-printer
fig, ax = plt.subplots(figsize=(7, 4.2))
bars(ax, list(cross.keys()), [c['acc'] for c in cross.values()],
     ['#D55E00'] * len(cross))
ax.set_ylim(0, max([c['acc'] for c in cross.values()]) + 12)
save(fig, 'fig6_crossprinter')

# fig7 — NEW: duration confound
fig, ax = plt.subplots(figsize=(7, 4.2))
v7 = [vib[0], RESULTS['duration_only_acc'], RESULTS['vib_duration_matched']]
bars(ax, ['Vibration\n(as submitted)', 'Duration\nfeatures ONLY',
          'Vibration\n(duration-matched)'], v7,
     ['#009E73', '#D55E00', '#0072B2'])
ax.set_ylim(0, max(v7) + 12)
save(fig, 'fig7_duration_confound')

# fig8 — NEW: acoustic window sensitivity
fig, ax = plt.subplots(figsize=(7, 4.2))
ks = [f'{k}s' for k in win_sweep] + [f'{N_MULTIWIN}x{ACOUSTIC_CAP}s\nspread']
vs = list(win_sweep.values()) + [RESULTS['acoustic_multiwindow']]
bars(ax, ks, vs, ['#0072B2'] * len(win_sweep) + ['#CC79A7'])
ax.set_ylim(0, max(vs + [chance]) + 15)
ax.set_xlabel('Audio analysed per recording')
save(fig, 'fig8_acoustic_window_sensitivity')


# ----------------------------------------------------------------------------
# 5. PROVENANCE OUTPUTS — this is what stops text and figures diverging again
# ----------------------------------------------------------------------------
RESULTS['config'] = {'B_PERM': B_PERM, 'SEEDS': SEEDS,
                     'ACOUSTIC_CAP': ACOUSTIC_CAP, 'SEG_SAMPLES': SEG_SAMPLES,
                     'T_PROP': T_PROP, 'RANDOM_STATE': RANDOM_STATE,
                     'sklearn_cv': 'StratifiedGroupKFold' if grouping_informative
                                   else 'StratifiedKFold'}
RESULTS['runtime_min'] = round((time.time() - t_start) / 60, 1)

with open(f'{OUT}/results.json', 'w') as f:
    json.dump(RESULTS, f, indent=2, default=str)

flat = {k: v for k, v in RESULTS.items()
        if isinstance(v, (int, float, str, bool)) or v is None}
pd.DataFrame([flat]).T.to_csv(f'{OUT}/results.csv', header=['value'])


def tex_cmd(name, val):
    if isinstance(val, float):
        val = f"{val:.2f}"
    return "\\newcommand{\\" + name + "}{" + str(val) + "}\n"


tex = ("% AUTO-GENERATED -- do not edit by hand.\n"
       "% \\input{paper_numbers.tex} in the preamble, then use \\AcAMNC etc.\n"
       "% Every number in the text AND every figure now comes from this run.\n")
tex += tex_cmd("AcAMNC",        RESULTS['acoustic_amnc'])
tex += tex_cmd("AcAMNCloCI",    RESULTS['acoustic_amnc_ci'][0])
tex += tex_cmd("AcAMNChiCI",    RESULTS['acoustic_amnc_ci'][1])
tex += tex_cmd("AcNotch",       RESULTS['acoustic_simnotch'])
tex += tex_cmd("AcPermP",       RESULTS['acoustic_perm_p'])
tex += tex_cmd("AcMultiWin",    RESULTS['acoustic_multiwindow'])
tex += tex_cmd("VibPooled",     RESULTS['vib_pooled'])
tex += tex_cmd("VibPooledloCI", RESULTS['vib_pooled_ci'][0])
tex += tex_cmd("VibPooledhiCI", RESULTS['vib_pooled_ci'][1])
tex += tex_cmd("VibPermP",      RESULTS['vib_perm_p'])
tex += tex_cmd("VibAmp",        RESULTS['vib_amplitude_only'])
tex += tex_cmd("VibFreq",       RESULTS['vib_frequency_only'])
tex += tex_cmd("VibStable",     RESULTS['vib_lenstable_only'])
tex += tex_cmd("DurOnly",       RESULTS['duration_only_acc'])
tex += tex_cmd("VibDurMatched", RESULTS['vib_duration_matched'])
tex += tex_cmd("TempOrdered",   temporal_ord[0])
tex += tex_cmd("TempOrderedSD", temporal_ord[1])
tex += tex_cmd("TempShuffled",  temporal_shf[0])
tex += tex_cmd("TempShuffledSD", temporal_shf[1])
tex += tex_cmd("NPerm",         B_PERM)
tex += tex_cmd("NSamples",      RESULTS['n_kept'])
tex += tex_cmd("Chance",        RESULTS['chance'])
Path(f'{OUT}/paper_numbers.tex').write_text(tex)

md = f"""# Analysis evidence summary
Run: {time.strftime('%Y-%m-%d %H:%M')} | runtime {RESULTS['runtime_min']} min
Samples {RESULTS['n_kept']} | classes {ncls} | chance {chance:.2f}%
Permutations B={B_PERM}, p=(r+1)/(B+1), floor {1/(B_PERM+1):.4f}
CV: {RESULTS['config']['sklearn_cv']} (grouping informative: {grouping_informative})

| Question | Evidence produced | Value |
|---|---|---|
| R1-1 no AMNC on/off | acoustic under AMNC + permutation null | {RESULTS['acoustic_amnc']}%, p={RESULTS['acoustic_perm_p']} |
| R1-1 (cont.) | external non-AMNC contrast | {RESULTS.get('external_nonamnc_acoustic', 'NOT RUN — set EXTERNAL_NONAMNC_DIR')} |
| R1-2 durations vary | duration-ONLY classifier | {RESULTS['duration_only_acc']}% |
| R1-2 (cont.) | duration-MATCHED vibration | {RESULTS['vib_duration_matched']}% vs {RESULTS['vib_pooled']}% |
| R1-2 (cont.) | fixed-length temporal, ordered vs shuffled | {temporal_ord[0]}% vs {temporal_shf[0]}% |
| R1-3 text/figure mismatch | paper_numbers.tex; figures read the same dict | see paper_numbers.tex |
| R1-4 sensor placement | dataset_metadata_dump.md | {len(RESULTS['metadata_files'])} doc files |
| R1-Q why 30 s | window sweep + whole-print sampling | {RESULTS['acoustic_window_sweep']}, multiwin {RESULTS['acoustic_multiwindow']}% |
| R1 permutation detail | B={B_PERM}, (r+1)/(B+1) | vib p={RESULTS['vib_perm_p']} |
| R2 generalisation | cross-printer + all Wilson CIs | {[f"{k} {v['acc']}%" for k, v in cross.items()]} |

## Decisions this run forces on you
1. If `duration_only_acc` is close to `vib_pooled`, the summary-feature result
   is substantially a duration artefact. Disclose it and lead with the
   duration-matched number instead.
2. If `fixed_ordered` - `fixed_shuffled` stays large, the sequential claim
   survives R1-2 and becomes your strongest, cleanest result. Make it primary.
3. If every entry in `acoustic_window_sweep` sits at chance, the 30 s cap is
   fully defensible — say so and cite this sweep.
4. If `grouping_informative` is False, remove every claim that grouped CV
   guards against leakage; describe it as stratified k-fold instead.
"""
Path(f'{OUT}/ANALYSIS_EVIDENCE.md').write_text(md)

print("\n" + "=" * 78)
print("DONE —", RESULTS['runtime_min'], "min")
print("=" * 78)
for f in sorted(os.listdir(OUT)):
    print("   ", f)
print("\nPaste-ready numbers:")
print(json.dumps({k: v for k, v in RESULTS.items()
                  if k not in ('metadata_files',)}, indent=2, default=str))

Mounted at /content/drive
Dataset root : /content/drive/MyDrive/3dprinter_sidechannel
Output root  : /content/drive/MyDrive/3dprinter_sidechannel/revision_v3

A0  DATASET METADATA (sensor placement evidence)
metadata/doc files found: 5
    TableOfContents.pdf
    output/fig4_confusion_col1.pdf
    output/fig7_perobject_col1.pdf
    output/fig8_ablation_classifiers_col1.pdf
    output/fig10_feature_importance.pdf

-> /content/drive/MyDrive/3dprinter_sidechannel/revision_v3/dataset_metadata_dump.md
   Read it. Whatever it does not specify, declare as UNKNOWN in the paper.

1  CATALOG
paired recordings 144 | classes 12 | printers 2 | sessions 144
samples per session: min 1 max 1 mean 1.00
GROUPING IS VACUOUS -- every session has one sample, so GroupKFold == KFold.
   Do not claim grouped CV as a validity safeguard in the paper if this prints.

2  FEATURE EXTRACTION
   0/144
   20/144
   40/144
   60/144
   80/144
   100/144
   120/144
   140/144
kept 144 samples | 12 classes | chance 8.33

---
## Stage 2 · Main analysis

Duration, vibration under equalised observation, feature ablation in hertz,
per-capture-configuration and per-printer results, cross-printer transfer, and the
temporal model with an order-shuffle control.

Key results: duration alone **63.89%**; vibration 22.22% time-matched; amplitude
26.39% vs frequency 9.03%; iPhone 25.00% ≈ Teensy 26.39%; P1P 36.11% vs A1 Mini
13.89%; temporal 25.42% ordered vs 15.28% shuffled.


In [ ]:
# =============================================================================
#  STAGE 2 — MAIN ANALYSIS
#
#  Catalog sanity check (aborts unless 144 / 12 / 2), per-recording sampling-rate
#  measurement, observation windows equalised in SECONDS, frequency features in
#  hertz, and results reported pooled, per capture configuration, and per printer.
# =============================================================================

!pip install -q pymupdf scikit-learn statsmodels

import os, re, json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# ============================== CONFIG =======================================
DRIVE_BASE = '/content/drive/MyDrive/3dprinter_sidechannel'
OUT        = '/content/drive/MyDrive/3dprinter_sidechannel/revision_v5'
T_SEG      = 120
RANDOM_STATE = 42
SEEDS      = [0, 1, 2, 3, 4]
EXPECT     = dict(n=144, classes=12, printers=2)   # abort if violated
ABORT_ON_MISMATCH = True
# =============================================================================

os.makedirs(OUT, exist_ok=True)
np.random.seed(RANDOM_STATE)
t0 = time.time()
RES = {}
BASE = Path(DRIVE_BASE)

# ---- BUG A FIX -------------------------------------------------------------
SKIP = {'results', 'results_v2', 'revision_v3', 'revision_v4', 'revision_v5',
        'output', 'Printing Videos', 'Noise Recordings', 'Artifacts',
        '__MACOSX', 'figures'}
AUD = {'.mp3', '.caf', '.wav', '.m4a'}

plt.rcParams.update({'font.size': 12, 'font.family': 'serif',
                     'figure.dpi': 300, 'savefig.dpi': 300,
                     'savefig.bbox': 'tight', 'axes.grid': True,
                     'grid.alpha': 0.3, 'grid.linestyle': '--'})


def wilson(a, n):
    if not n:
        return (float('nan'),) * 2
    lo, hi = proportion_confint(int(round(a / 100 * n)), n, 0.05, 'wilson')
    return round(lo * 100, 2), round(hi * 100, 2)


def skipped(p):
    return any(s in p.parts for s in SKIP)


# =============================== CATALOG =====================================
print("=" * 78)
print("CATALOG")
print("=" * 78)
rows = []
for obj in sorted(BASE.iterdir()):
    if not obj.is_dir() or obj.name in SKIP or obj.name.endswith('.zip'):
        continue
    if not re.match(r'^\d\d_', obj.name):          # class folders only
        continue
    nested = sorted([d for d in obj.iterdir() if d.is_dir()])
    if not nested:
        continue
    for pr in sorted(nested[0].iterdir()):
        if not pr.is_dir() or pr.name in SKIP:
            continue
        sess = {}
        for f in sorted(pr.rglob('*')):
            e = f.suffix.lower()
            if (e in AUD or e == '.csv') and not skipped(f):
                s = sess.setdefault(f.parent, {'a': [], 'c': []})
                (s['a'] if e in AUD else s['c']).append(f)
        for folder in sorted(sess):
            fs_ = sess[folder]
            if fs_['a'] and fs_['c']:
                for af in sorted(fs_['a']):
                    rows.append({'label': obj.name, 'printer': pr.name,
                                 'audio': str(af),
                                 'vibr': str(sorted(fs_['c'])[0])})
cat = (pd.DataFrame(rows).sort_values(['label', 'printer', 'audio'])
         .reset_index(drop=True))

print(f"recordings {len(cat)} | classes {cat['label'].nunique()} | "
      f"printers {cat['printer'].nunique()}")
print("printer folders:", sorted(cat['printer'].unique()))
print("classes:", sorted(cat['label'].unique()))

bad = []
if len(cat) != EXPECT['n']:
    bad.append(f"expected {EXPECT['n']} recordings, got {len(cat)}")
if cat['label'].nunique() != EXPECT['classes']:
    bad.append(f"expected {EXPECT['classes']} classes, got {cat['label'].nunique()}")
if cat['printer'].nunique() != EXPECT['printers']:
    bad.append(f"expected {EXPECT['printers']} printers, got {cat['printer'].nunique()}")
if bad:
    print("\n*** CATALOG SANITY CHECK FAILED ***")
    for b in bad:
        print("   ", b)
    if ABORT_ON_MISMATCH:
        raise SystemExit("Catalog wrong — fix SKIP/paths before trusting anything below.")
else:
    print("\nCATALOG OK — 144 / 12 / 2 as expected.")


# ========================= SAMPLE RATE + SYSTEM ==============================
print("\n" + "=" * 78)
print("SAMPLE RATE AND CAPTURE SYSTEM")
print("=" * 78)
TCOLS = ('t', 'time', 'timestamp', 'millis', 'micros', 'ms', 'us', 'elapsed')
DOC_RATES = (100.0, 200.0, 500.0)


def detect_fs(tv):
    d = np.median(np.diff(tv))
    if not np.isfinite(d) or d <= 0:
        return None, None
    best = None
    for scale, name in ((1e-9, 'ns'), (1e-6, 'us'), (1e-3, 'ms'), (1.0, 's')):
        fs = 1.0 / (d * scale)
        if 1.0 <= fs <= 20000.0:
            err = min(abs(np.log(fs / r)) for r in DOC_RATES)
            if best is None or err < best[2]:
                best = (fs, name, err)
    return (best[0], best[1]) if best else (None, None)


def load_rec(p):
    try:
        df = pd.read_csv(p)
    except Exception:
        return None, None, None
    df.columns = [str(c).strip().lower() for c in df.columns]
    if not {'x', 'y', 'z'} <= set(df.columns):
        return None, None, None
    a = df[['x', 'y', 'z']].apply(pd.to_numeric, errors='coerce').dropna().to_numpy(float)
    if len(a) < 100:
        return None, None, None
    fs = unit = None
    for tc in TCOLS:
        if tc in df.columns:
            tv = pd.to_numeric(df[tc], errors='coerce').dropna().to_numpy(float)
            if len(tv) > 10:
                fs, unit = detect_fs(tv)
            break
    return a, fs, unit


print("reading...")
raw, meta = [], []
for i, (_, r) in enumerate(cat.iterrows()):
    if i % 30 == 0:
        print(f"  {i}/{len(cat)}", flush=True)
    a, fs, unit = load_rec(r['vibr'])
    if a is None or not fs:
        continue
    raw.append(a)
    meta.append({'label': r['label'], 'printer': r['printer'], 'n': len(a),
                 'fs': fs, 'unit': unit, 'dur_s': len(a) / fs})
M = pd.DataFrame(meta)
M['system'] = np.where(M['fs'] < 350, 'iPhone', 'Teensy')

print(f"\nloaded {len(M)}")
print("time units:", M['unit'].value_counts().to_dict())
print(M.groupby('system')['fs'].agg(['count', 'median', 'min', 'max']).round(1).to_string())
xt = pd.crosstab(M['label'], M['system'])
print("\nsystem x class:")
print(xt.to_string())
balanced = bool(xt.shape[1] < 2 or (xt.min(axis=1) > 0).all())
print("\nBALANCED across classes." if balanced else "\n** CONFOUND **")
RES.update(fs_by_system=M.groupby('system')['fs'].median().round(2).to_dict(),
           units=M['unit'].value_counts().to_dict(),
           system_counts=M['system'].value_counts().to_dict(),
           system_balanced=balanced,
           nyquist=[round(float(M['fs'].min() / 2), 1),
                    round(float(M['fs'].max() / 2), 1)])
print(f"Nyquist {RES['nyquist'][0]}-{RES['nyquist'][1]} Hz")


# ==================== BUG B FIX: G-CODE GROUND TRUTH =========================
print("\n" + "=" * 78)
print("G-CODE GROUND TRUTH")
print("=" * 78)
PATS = [r'estimated printing time.*?=\s*([0-9hmsd\s]+)',
        r'model printing time:\s*([0-9hmsd\s]+)',
        r'total estimated time:\s*([0-9hmsd\s]+)',
        r';TIME:\s*([0-9.]+)']


def hms(s):
    s = s.strip().lower()
    if re.fullmatch(r'[0-9.]+', s):
        return float(s)
    tot = 0.0
    hit = False
    for v, u in re.findall(r'([0-9.]+)\s*([dhms])', s):
        tot += float(v) * {'d': 86400, 'h': 3600, 'm': 60, 's': 1}[u]
        hit = True
    return tot if hit else None


def gtime(p):
    try:
        head = "".join(l for i, l in zip(range(400), open(p, errors='ignore')))
        with open(p, 'rb') as f:
            f.seek(max(0, os.path.getsize(p) - 200000))
            tail = f.read().decode('utf-8', errors='ignore')
        for blob in (head, tail):
            for pat in PATS:
                m = re.search(pat, blob, re.I)
                if m:
                    v = hms(m.group(1))
                    if v and v > 0:
                        return v
    except Exception:
        pass
    return None


# key on the leading NN_ prefix, which both class folders and g-code files share
prefix2label = {l[:3]: l for l in sorted(M['label'].unique())}
gmodel, gfile = {}, {}
for p in BASE.rglob('*.gcode'):
    if skipped(p):
        continue
    pre = p.name[:3]
    lab = prefix2label.get(pre)
    if not lab:
        continue
    t = gtime(p)
    if t:
        gmodel.setdefault(lab, []).append(t)
    fm = re.search(r'_((?:\d+h)?(?:\d+m)?(?:\d+s)?)\.gcode$', p.name)
    if fm and fm.group(1):
        tv = hms(fm.group(1))
        if tv:
            gfile.setdefault(lab, []).append(tv)

gm = {k: float(np.median(v)) for k, v in gmodel.items()}
gf = {k: float(np.median(v)) for k, v in gfile.items()}
print(f"model-time parsed for {len(gm)} classes; filename-time for {len(gf)}")
if gm and gf:
    common = sorted(set(gm) & set(gf))
    off = [gf[k] - gm[k] for k in common]
    print(f"filename-minus-model offset: median {np.median(off)/60:.2f} min "
          f"(startup overhead), sd {np.std(off)/60:.2f} min")

M['gcode_s'] = M['label'].map(gm)
M['gcode_file_s'] = M['label'].map(gf)
ok = M['gcode_s'].notna()
print(f"mapped to {int(ok.sum())}/{len(M)} recordings")
if ok.sum() > 10:
    r1 = float(np.corrcoef(M.loc[ok, 'dur_s'], M.loc[ok, 'gcode_s'])[0, 1])
    RES['corr_recording_vs_gcode'] = round(r1, 4)
    print(f"\ncorr(recording duration, g-code model time) r = {r1:.4f}")
    if r1 > 0.9:
        print("  => recording length tracks TRUE print time (ground truth).")
        print("     The duration finding is confirmed against g-code, not")
        print("     inferred from file length. This is the citable version.")
    else:
        print("  => recording length does NOT track true print time.")
    ok2 = M['gcode_file_s'].notna()
    if ok2.sum() > 10:
        r2 = float(np.corrcoef(M.loc[ok2, 'dur_s'], M.loc[ok2, 'gcode_file_s'])[0, 1])
        RES['corr_recording_vs_filename'] = round(r2, 4)
        print(f"corr(recording duration, filename time)   r = {r2:.4f}")


# ============================ FEATURES =======================================
print("\n" + "=" * 78)
print("FEATURES — time-matched span, frequency in Hz")
print("=" * 78)
T_SEC = float(M['dur_s'].min())
print(f"shortest recording {T_SEC:.1f} s -> all truncated to {T_SEC:.1f} SECONDS")
RES['matched_seconds'] = round(T_SEC, 2)
print(f"duration range: {M['dur_s'].min():.1f}-{M['dur_s'].max():.1f} s")


def summ(a, fs):
    out = []
    for ax in range(3):
        v = a[:, ax]
        mag = np.abs(np.fft.rfft(v - v.mean()))
        frq = np.fft.rfftfreq(len(v), d=1.0 / fs)
        out += [v.mean(), v.std(), np.sqrt((v ** 2).mean()),
                v.max() - v.min(), frq[mag.argmax()]]
    return np.array(out)


Xv, Xvt = [], []
for a, (_, m) in zip(raw, M.iterrows()):
    Xv.append(summ(a, m['fs']))
    Xvt.append(summ(a[:int(T_SEC * m['fs'])], m['fs']))
Xv = np.nan_to_num(np.vstack(Xv))
Xvt = np.nan_to_num(np.vstack(Xvt))

le = LabelEncoder().fit(M['label'])
y = le.transform(M['label'])
ncls = len(le.classes_)
chance = 100 / ncls
sysv = M['system'].to_numpy()
prv = M['printer'].to_numpy()
Xdur = np.c_[M['dur_s'].to_numpy(), M['n'].to_numpy()]
print(f"classes {ncls} | chance {chance:.2f}%")


def rf():
    return Pipeline([('s', StandardScaler()),
                     ('m', RandomForestClassifier(200, random_state=RANDOM_STATE,
                                                  n_jobs=-1))])


def acc(X, yy, folds=5):
    n_min = pd.Series(yy).value_counts().min()
    k = int(min(folds, n_min))
    if k < 2:
        return None, None
    p = cross_val_predict(rf(), np.nan_to_num(X), yy,
                          cv=StratifiedKFold(k, shuffle=True,
                                             random_state=RANDOM_STATE))
    return round(accuracy_score(yy, p) * 100, 2), p


# --------------------------- POOLED ------------------------------------------
print("\n--- POOLED (both capture systems) ---")
a_vf, _ = acc(Xv, y)
a_vt, _ = acc(Xvt, y)
a_du, p_du = acc(Xdur, y)
a_bo, p_bo = acc(np.hstack([Xdur, Xvt]), y)
amp_i = [0, 1, 2, 3, 5, 6, 7, 8, 10, 11, 12, 13]
frq_i = [4, 9, 14]
a_am, _ = acc(Xvt[:, amp_i], y)
a_fr, _ = acc(Xvt[:, frq_i], y)
for nm, v in [('vibration full', a_vf), (f'vibration {T_SEC:.0f}s-matched', a_vt),
              ('  amplitude only', a_am), ('  frequency (Hz) only', a_fr),
              ('duration only', a_du), ('duration + vibration', a_bo)]:
    print(f"  {nm:28s} {v:6.2f}%  CI {wilson(v, len(y))}")
RES.update(pooled=dict(vib_full=a_vf, vib_matched=a_vt, amplitude=a_am,
                       frequency_hz=a_fr, duration=a_du, dur_plus_vib=a_bo,
                       duration_ci=wilson(a_du, len(y)),
                       vib_matched_ci=wilson(a_vt, len(y))))
cd, cb = (p_du == y).astype(int), (p_bo == y).astype(int)
tb = [[int(((cd == 1) & (cb == 1)).sum()), int(((cd == 1) & (cb == 0)).sum())],
      [int(((cd == 0) & (cb == 1)).sum()), int(((cd == 0) & (cb == 0)).sum())]]
RES['mcnemar_p'] = round(float(mcnemar(tb, exact=True).pvalue), 6)
RES['mcnemar_table'] = tb
print(f"  McNemar p = {RES['mcnemar_p']}   {tb}")

# can the features identify the capture rig?
a_sys, _ = acc(Xvt, LabelEncoder().fit_transform(sysv))
RES['sensor_id_accuracy'] = a_sys
print(f"\n  capture system identifiable from vibration: {a_sys:.2f}% "
      f"(chance {100/M['system'].nunique():.1f}%)")

# --------------------- PER CAPTURE SYSTEM ------------------------------------
print("\n--- PER CAPTURE SYSTEM (v4 showed these are separable) ---")
RES['per_system'] = {}
for s in sorted(M['system'].unique()):
    m = sysv == s
    d = {}
    d['n'] = int(m.sum())
    d['vib_matched'], _ = acc(Xvt[m], y[m])
    d['duration'], pdu = acc(Xdur[m], y[m])
    d['dur_plus_vib'], pbo = acc(np.hstack([Xdur, Xvt])[m], y[m])
    d['amplitude'], _ = acc(Xvt[m][:, amp_i], y[m])
    d['frequency_hz'], _ = acc(Xvt[m][:, frq_i], y[m])
    d['vib_ci'] = wilson(d['vib_matched'], d['n'])
    d['dur_ci'] = wilson(d['duration'], d['n'])
    RES['per_system'][s] = d
    print(f"  {s:8s} n={d['n']:3d} | vib {d['vib_matched']:6.2f}% "
          f"CI {d['vib_ci']} | dur {d['duration']:6.2f}% CI {d['dur_ci']} "
          f"| both {d['dur_plus_vib']:6.2f}% | amp {d['amplitude']:6.2f}% "
          f"| freq {d['frequency_hz']:6.2f}%")

# --------------------- PER PRINTER -------------------------------------------
print("\n--- PER PRINTER ---")
RES['per_printer'] = {}
for p in sorted(M['printer'].unique()):
    m = prv == p
    a_, _ = acc(Xvt[m], y[m])
    RES['per_printer'][p] = {'n': int(m.sum()), 'vib_matched': a_,
                             'ci': wilson(a_, int(m.sum()))}
    print(f"  {p:16s} n={int(m.sum()):3d}  vib {a_:6.2f}%  "
          f"CI {RES['per_printer'][p]['ci']}")

# --------------------- CROSS-PRINTER -----------------------------------------
print("\n--- CROSS-PRINTER TRANSFER ---")
RES['cross_printer'] = {}
ps = sorted(M['printer'].unique())
for a_p in ps:
    for b_p in ps:
        if a_p == b_p:
            continue
        mdl = rf().fit(Xvt[prv == a_p], y[prv == a_p])
        v = accuracy_score(y[prv == b_p], mdl.predict(Xvt[prv == b_p])) * 100
        RES['cross_printer'][f'{a_p}->{b_p}'] = {
            'acc': round(v, 2), 'ci': wilson(v, int((prv == b_p).sum()))}
        print(f"  {a_p} -> {b_p}: {v:.2f}%")


# ============================ TEMPORAL =======================================
print("\n" + "=" * 78)
print("TEMPORAL — equal seconds per segment")
print("=" * 78)
try:
    import torch
    import torch.nn as nn
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'

    seqs = []
    for a, (_, m) in zip(raw, M.iterrows()):
        k = int(T_SEC * m['fs'])
        seg = max(8, k // T_SEG)
        rr = []
        for i in range(T_SEG):
            w = a[i * seg:(i + 1) * seg]
            if len(w) < 4:
                w = a[:4]
            row = []
            for ax in range(3):
                v = w[:, ax]
                mg = np.abs(np.fft.rfft(v - v.mean()))
                fq = np.fft.rfftfreq(len(v), d=1.0 / m['fs'])
                row += [v.std(), np.sqrt((v ** 2).mean()), v.max() - v.min(),
                        fq[mg.argmax()]]
            rr.append(row)
        seqs.append(np.asarray(rr, np.float32))
    S = np.stack(seqs)
    print(f"{S.shape} | {T_SEC/T_SEG:.2f} s per segment")

    class Net(nn.Module):
        def __init__(s, n):
            super().__init__()
            s.f = nn.Sequential(
                nn.Conv1d(12, 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU(),
                nn.Conv1d(32, 32, 5, padding=4, dilation=2), nn.BatchNorm1d(32), nn.ReLU(),
                nn.Conv1d(32, 64, 5, padding=8, dilation=4), nn.BatchNorm1d(64), nn.ReLU(),
                nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, n))

        def forward(s, x):
            return s.f(x)

    def run(shuffle, tag):
        accs = []
        for sd in SEEDS:
            rng = np.random.default_rng(sd)
            X = []
            for s_ in S:
                z = (s_ - s_.mean(0)) / (s_.std(0) + 1e-6)
                if shuffle:
                    z = z[rng.permutation(len(z))]
                X.append(z.T.astype(np.float32))
            X = np.stack(X)
            pred = np.zeros(len(y), int)
            for tr, te in StratifiedKFold(5, shuffle=True, random_state=sd).split(X, y):
                torch.manual_seed(sd)
                mdl = Net(ncls).to(dev)
                opt = torch.optim.Adam(mdl.parameters(), 1e-3, weight_decay=1e-3)
                lf = nn.CrossEntropyLoss()
                Xt = torch.tensor(X[tr]).to(dev)
                yt = torch.tensor(y[tr]).long().to(dev)
                mdl.train()
                for _ in range(40):
                    for idx in torch.randperm(len(Xt)).split(32):
                        if len(idx) < 2:
                            continue
                        opt.zero_grad()
                        lf(mdl(Xt[idx]), yt[idx]).backward()
                        opt.step()
                mdl.eval()
                with torch.no_grad():
                    pred[te] = mdl(torch.tensor(X[te]).to(dev)).argmax(1).cpu().numpy()
            accs.append(accuracy_score(y, pred) * 100)
        m_, s_ = round(float(np.mean(accs)), 2), round(float(np.std(accs)), 2)
        print(f"  {tag:28s} {m_:5.2f}% +/- {s_:.2f}  {np.round(accs,2)}")
        return m_, s_

    o = run(False, "ORDERED")
    sh = run(True, "SHUFFLED")
    RES.update(temporal_ordered=o, temporal_shuffled=sh,
               temporal_gap=round(o[0] - sh[0], 2))
    print(f"\n  gap {RES['temporal_gap']} points over {len(SEEDS)} seeds")
except Exception as e:
    print("temporal failed:", e)
    RES['temporal_error'] = str(e)


# ============================== OUTPUT =======================================
fig, ax = plt.subplots(figsize=(8.5, 4.3))
v = [RES['pooled']['duration'], RES['pooled']['vib_matched'],
     RES['pooled']['dur_plus_vib'],
     RES.get('temporal_ordered', [np.nan])[0],
     RES.get('temporal_shuffled', [np.nan])[0]]
lb = ['Duration\nonly', f'Vibration\n({T_SEC:.0f}s)', 'Duration +\nvibration',
      'Temporal\nordered', 'Temporal\nshuffled']
b = ax.bar(lb, v, color=['#D55E00', '#0072B2', '#009E73', '#E69F00', '#999999'],
           edgecolor='black')
for bb, vv in zip(b, v):
    if np.isfinite(vv):
        ax.text(bb.get_x() + bb.get_width() / 2, vv + 1, f'{vv:.1f}%',
                ha='center', fontsize=9)
ax.axhline(chance, color='red', ls='--', label=f'Chance ({chance:.1f}%)')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, np.nanmax(v) + 12)
ax.legend()
fig.tight_layout()
for e in ('png', 'pdf'):
    fig.savefig(f'{OUT}/fig11_final.{e}')
plt.close(fig)

M.to_csv(f'{OUT}/recording_metadata.csv', index=False)
RES['runtime_min'] = round((time.time() - t0) / 60, 1)
RES['n'] = int(len(M))
RES['chance'] = round(chance, 2)
Path(f'{OUT}/results_v5.json').write_text(json.dumps(RES, indent=2, default=str))

print("\n" + "=" * 78)
print(f"DONE — {RES['runtime_min']} min  ->  {OUT}")
print("=" * 78)
print(json.dumps(RES, indent=2, default=str))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CATALOG
recordings 144 | classes 12 | printers 2
printer folders: ['Bambu_A1mini', 'Bambu_P1P']
classes: ['01_key_easy', '02_key_medium', '03_key_hard', '04_key_steps', '05_2keys', '06_Autodesk_kickstarter_FDM_test', '07_NIST_additive_test', '08_ASTM', '09_All_in_1', '10_Triple_Helix', '11_CaliCat', '12_retraction_test']

CATALOG OK — 144 / 12 / 2 as expected.

SAMPLE RATE AND CAPTURE SYSTEM
reading...
  0/144
  30/144
  60/144
  90/144
  120/144

loaded 144
time units: {'s': 72, 'ms': 72}
        count  median    min    max
system                             
Teensy     72   500.0  500.0  500.0
iPhone     72   199.6  198.9  199.8

system x class:
system                            Teensy  iPhone
label                                           
01_key_easy                            6       6
02_key_medium                          6       6
03_key_hard        

---
## Stage 3 · G-code ground truth

Parses estimated print time from the sliced G-code shipped with the dataset and
correlates it against recording duration. This is what makes the duration finding a
claim about the physical process rather than about file lengths.

Key result: Pearson **r = 0.9073**, Spearman ρ = 0.8933, n = 144.


In [ ]:
# =============================================================================
#  STAGE 3 — G-CODE GROUND TRUTH
#
#  The .gcode files live under Artifacts/, which the analysis catalog deliberately
#  excludes, so this stage searches for them with a narrower skip list. Reads the
#  metadata written by Stage 2.
# =============================================================================

import os, re, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DRIVE_BASE = '/content/drive/MyDrive/3dprinter_sidechannel'
OUT        = '/content/drive/MyDrive/3dprinter_sidechannel/revision_v5'
BASE = Path(DRIVE_BASE)

# Only exclude output folders here — NOT Artifacts, which is where g-code lives.
GSKIP = {'results', 'results_v2', 'revision_v3', 'revision_v4', 'revision_v5',
         'output', '__MACOSX', 'figures'}

M = pd.read_csv(f'{OUT}/recording_metadata.csv')
print(f"metadata: {len(M)} recordings, {M['label'].nunique()} classes")

# ---------------------------------------------------------------- find files
cands = []
for pat in ('*.gcode', '*.gco', '*.g'):
    cands += [p for p in BASE.rglob(pat)
              if not any(s in p.parts for s in GSKIP)]
print(f"g-code files found: {len(cands)}")
for p in cands[:30]:
    print("   ", p.relative_to(BASE))

# also look inside 3mf archives (they are zips containing gcode)
import zipfile
mf = [p for p in BASE.rglob('*.3mf') if not any(s in p.parts for s in GSKIP)]
print(f".3mf archives found: {len(mf)}")

# ------------------------------------------------------------------- parsing
PATS = [r'estimated printing time.*?=\s*([0-9hmsd\s]+)',
        r'model printing time:\s*([0-9hmsd\s]+)',
        r'total estimated time:\s*([0-9hmsd\s]+)',
        r';TIME:\s*([0-9.]+)']


def hms(s):
    s = str(s).strip().lower()
    if re.fullmatch(r'[0-9.]+', s):
        return float(s)
    tot, hit = 0.0, False
    for v, u in re.findall(r'([0-9.]+)\s*([dhms])', s):
        tot += float(v) * {'d': 86400, 'h': 3600, 'm': 60, 's': 1}[u]
        hit = True
    return tot if hit else None


def scan_text(blob):
    for pat in PATS:
        m = re.search(pat, blob, re.I)
        if m:
            v = hms(m.group(1))
            if v and v > 0:
                return v
    return None


def gtime_file(p):
    try:
        head = "".join(l for i, l in zip(range(600), open(p, errors='ignore')))
        with open(p, 'rb') as f:
            f.seek(max(0, os.path.getsize(p) - 300000))
            tail = f.read().decode('utf-8', errors='ignore')
        return scan_text(head) or scan_text(tail)
    except Exception:
        return None


labels = sorted(M['label'].unique())
pre2lab = {l[:3]: l for l in labels}
model_t, file_t, recs = {}, {}, []

for p in cands:
    lab = pre2lab.get(p.name[:3])
    if lab is None:                       # fall back to path parts
        lab = next((pre2lab.get(part[:3]) for part in p.parts
                    if pre2lab.get(part[:3])), None)
    if lab is None:
        continue
    t = gtime_file(p)
    fm = re.search(r'_((?:\d+h)?(?:\d+m)?(?:\d+s)?)\.g(?:code|co)?$', p.name)
    ft = hms(fm.group(1)) if (fm and fm.group(1)) else None
    if t:
        model_t.setdefault(lab, []).append(t)
    if ft:
        file_t.setdefault(lab, []).append(ft)
    recs.append({'label': lab, 'file': p.name, 'model_s': t, 'filename_s': ft})

for p in mf:
    lab = pre2lab.get(p.name[:3]) or next(
        (pre2lab.get(part[:3]) for part in p.parts if pre2lab.get(part[:3])), None)
    if lab is None:
        continue
    try:
        with zipfile.ZipFile(p) as z:
            for nm in z.namelist():
                if nm.lower().endswith(('.gcode', '.gco')):
                    txt = z.read(nm)[:400000].decode('utf-8', errors='ignore')
                    t = scan_text(txt)
                    if t:
                        model_t.setdefault(lab, []).append(t)
                        recs.append({'label': lab, 'file': f'{p.name}:{nm}',
                                     'model_s': t, 'filename_s': None})
                    break
    except Exception:
        pass

G = pd.DataFrame(recs)
if len(G):
    G.to_csv(f'{OUT}/gcode_times.csv', index=False)

gm = {k: float(np.median(v)) for k, v in model_t.items()}
gf = {k: float(np.median(v)) for k, v in file_t.items()}
print(f"\nmodel-time parsed for {len(gm)} classes; filename-time for {len(gf)}")

if not gm and not gf:
    print("\nSTILL ZERO. Run this to see where g-code actually lives:")
    print("  !find '/content/drive/MyDrive/3dprinter_sidechannel' "
          "-iname '*.gcode' -o -iname '*.3mf' | head -40")
    raise SystemExit

for k in sorted(set(gm) | set(gf)):
    print(f"   {k:34s} model {gm.get(k, float('nan'))/60:8.1f} min   "
          f"filename {gf.get(k, float('nan'))/60:8.1f} min")

common = sorted(set(gm) & set(gf))
if common:
    off = [gf[k] - gm[k] for k in common]
    print(f"\nfilename - model offset: median {np.median(off)/60:.2f} min, "
          f"sd {np.std(off)/60:.2f} min  (startup overhead)")

# --------------------------------------------------------------- correlation
M['gcode_s'] = M['label'].map(gm)
M['gcode_file_s'] = M['label'].map(gf)
res = {}
for col, tag in (('gcode_s', 'g-code model time'),
                 ('gcode_file_s', 'filename time')):
    ok = M[col].notna()
    if ok.sum() > 10:
        r = float(np.corrcoef(M.loc[ok, 'dur_s'], M.loc[ok, col])[0, 1])
        rs = float(pd.Series(M.loc[ok, 'dur_s']).corr(
            pd.Series(M.loc[ok, col]), method='spearman'))
        res[f'corr_{col}_pearson'] = round(r, 4)
        res[f'corr_{col}_spearman'] = round(rs, 4)
        res[f'n_{col}'] = int(ok.sum())
        print(f"\ncorr(recording duration, {tag}): "
              f"Pearson r = {r:.4f} | Spearman rho = {rs:.4f}  (n={int(ok.sum())})")

pear = res.get('corr_gcode_s_pearson')
if pear is not None:
    print()
    if pear > 0.9:
        print("=> Recording length tracks TRUE print time from the sliced g-code.")
        print("   The duration result reflects genuine print duration, validated")
        print("   against ground truth rather than inferred from file length.")
        print("   Report the correlation in the paper — it is the strongest form")
        print("   of the duration finding.")
    else:
        print("=> Recording length does NOT track g-code print time.")
        print("   The duration feature is partly a recording artefact and the")
        print("   claim must be scoped to observed recording length only.")

# ------------------------------------------------------------------- figure
ok = M['gcode_s'].notna()
if ok.sum() > 10:
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.scatter(M.loc[ok, 'gcode_s'] / 60, M.loc[ok, 'dur_s'] / 60,
               s=26, alpha=0.75, edgecolor='black', linewidth=0.4,
               color='#0072B2')
    lim = [0, max(M.loc[ok, 'gcode_s'].max(), M.loc[ok, 'dur_s'].max()) / 60 * 1.05]
    ax.plot(lim, lim, 'r--', lw=1, label='y = x')
    ax.set_xlabel('G-code estimated print time (min)')
    ax.set_ylabel('Recording duration (min)')
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.legend()
    fig.tight_layout()
    for e in ('png', 'pdf'):
        fig.savefig(f'{OUT}/fig12_gcode_validation.{e}')
    plt.close(fig)
    print(f"\nsaved fig12_gcode_validation")

# ------------------------------------------------------------------- persist
rp = Path(f'{OUT}/results_v5.json')
allr = json.loads(rp.read_text()) if rp.exists() else {}
allr['gcode_validation'] = res
allr['gcode_model_times_s'] = {k: round(v, 1) for k, v in gm.items()}
rp.write_text(json.dumps(allr, indent=2, default=str))
print(f"\nappended to {rp}")
print(json.dumps(res, indent=2))

metadata: 144 recordings, 12 classes
g-code files found: 12
    Artifacts/Artifacts/P1P_prints/10_tri-helix-tube-105mm_PLA_1h28m.gcode
    Artifacts/Artifacts/P1P_prints/01_key_easy_P1P_10m56s.gcode
    Artifacts/Artifacts/P1P_prints/02_key_medium_P1P_10m56s.gcode
    Artifacts/Artifacts/P1P_prints/03_key_hard_P1P_10m54s.gcode
    Artifacts/Artifacts/P1P_prints/04_key_steps_P1P_10m54s.gcode
    Artifacts/Artifacts/P1P_prints/05_2keys_P1P_15m33s.gcode
    Artifacts/Artifacts/P1P_prints/06_ksr_fdmtest_v4_P1P_2h32m.gcode
    Artifacts/Artifacts/P1P_prints/07_NIST Test Artifact online_P1P_2h15m.gcode
    Artifacts/Artifacts/P1P_prints/08_ASTM_P1P_3h9m.gcode
    Artifacts/Artifacts/P1P_prints/09_3D_Printer_test_fixed_stl_3rd_gen_P1P_2h54m.gcode
    Artifacts/Artifacts/P1P_prints/11_calicat_PLA_P1P_39m2s.gcode
    Artifacts/Artifacts/P1P_prints/12_retraction_test_P1P_12m59s.gcode
.3mf archives found: 12

model-time parsed for 12 classes; filename-time for 12
   01_key_easy                   

---
## Stage 4 · Spectral verification, plus exploratory analyses

The reported result is the spectral comparison of print audio against the dataset's own
background noise recordings in the 120–360 Hz motor-resonance band. This is
classifier-independent evidence that suppression is operating, and the closest available
substitute for the on/off ablation the hardware does not permit.

Key result: motor band **+9.25 dB** during printing vs **+4.33 dB** in background —
a **+4.92 dB** elevation.

The cell also contains exploratory analyses (duration-stratified classification, window
position, per-configuration acoustic, incremental tests). These informed the design of
Stage 5 and are retained for transparency; **only the spectral result is reported in the


In [ ]:
# =============================================================================
#  STAGE 4 — SPECTRAL VERIFICATION (+ EXPLORATORY)
#
#  REPORTED: motor-band (120-360 Hz) power relative to a 600-2000 Hz reference,
#  print audio vs the dataset's background noise recordings.
#
#  EXPLORATORY, not carried into the summary: duration-stratified classification,
#  acoustic window position, per-configuration acoustic split, incremental tests.
#  These motivated Stage 5 and are kept for transparency.
# =============================================================================

!pip install -q librosa scikit-learn statsmodels

import os, re, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import librosa
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar

warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# ============================== CONFIG =======================================
DRIVE_BASE = '/content/drive/MyDrive/3dprinter_sidechannel'
OUT        = '/content/drive/MyDrive/3dprinter_sidechannel/revision_v6'
WIN        = 60           # duration-clean acoustic window (all recs >= 88.3 s)
RANDOM_STATE = 42
B_PERM     = 1000
# =============================================================================

os.makedirs(OUT, exist_ok=True)
np.random.seed(RANDOM_STATE)
t0 = time.time()
R = {}
BASE = Path(DRIVE_BASE)
SKIP = {'results', 'results_v2', 'revision_v3', 'revision_v4', 'revision_v5',
        'revision_v6', 'output', 'Printing Videos', 'Noise Recordings',
        'Artifacts', '__MACOSX', 'figures'}
AUD = {'.mp3', '.caf', '.wav', '.m4a'}

plt.rcParams.update({'font.size': 12, 'font.family': 'serif',
                     'figure.dpi': 300, 'savefig.dpi': 300,
                     'savefig.bbox': 'tight', 'axes.grid': True,
                     'grid.alpha': 0.3, 'grid.linestyle': '--'})


def wilson(a, n):
    if not n:
        return (float('nan'),) * 2
    lo, hi = proportion_confint(int(round(a / 100 * n)), n, 0.05, 'wilson')
    return round(lo * 100, 2), round(hi * 100, 2)


def rf():
    return Pipeline([('s', StandardScaler()),
                     ('m', RandomForestClassifier(200, random_state=RANDOM_STATE,
                                                  n_jobs=-1))])


def cvacc(X, yy, folds=5):
    n_min = pd.Series(yy).value_counts().min()
    k = int(min(folds, n_min))
    if k < 2 or len(set(yy)) < 2:
        return None, None
    p = cross_val_predict(rf(), np.nan_to_num(X), yy,
                          cv=StratifiedKFold(k, shuffle=True,
                                             random_state=RANDOM_STATE))
    return round(accuracy_score(yy, p) * 100, 2), p


def perm_p(X, yy, obs, B=B_PERM, tag=""):
    rng, r = np.random.default_rng(RANDOM_STATE), 0
    for b in range(B):
        if b % max(1, B // 5) == 0:
            print(f"      perm {tag} {b}/{B}", flush=True)
        a, _ = cvacc(X, rng.permutation(yy))
        if a is not None and a >= obs:
            r += 1
    return round((r + 1) / (B + 1), 5)


# =============================== CATALOG =====================================
rows = []
for obj in sorted(BASE.iterdir()):
    if not obj.is_dir() or obj.name in SKIP or not re.match(r'^\d\d_', obj.name):
        continue
    nested = sorted([d for d in obj.iterdir() if d.is_dir()])
    if not nested:
        continue
    for pr in sorted(nested[0].iterdir()):
        if not pr.is_dir() or pr.name in SKIP:
            continue
        sess = {}
        for f in sorted(pr.rglob('*')):
            e = f.suffix.lower()
            if e in AUD or e == '.csv':
                s = sess.setdefault(f.parent, {'a': [], 'c': []})
                (s['a'] if e in AUD else s['c']).append(f)
        for folder in sorted(sess):
            fs_ = sess[folder]
            if fs_['a'] and fs_['c']:
                for af in sorted(fs_['a']):
                    rows.append({'label': obj.name, 'printer': pr.name,
                                 'audio': str(af), 'audio_ext': af.suffix.lower(),
                                 'vibr': str(sorted(fs_['c'])[0])})
cat = (pd.DataFrame(rows).sort_values(['label', 'printer', 'audio'])
         .reset_index(drop=True))
print(f"catalog: {len(cat)} recordings | {cat['label'].nunique()} classes")
assert len(cat) == 144 and cat['label'].nunique() == 12, "CATALOG WRONG — stop"


# ============================== LOAD =========================================
TCOLS = ('t', 'time', 'timestamp', 'millis', 'micros', 'ms', 'us', 'elapsed')


def detect_fs(tv):
    d = np.median(np.diff(tv))
    if not np.isfinite(d) or d <= 0:
        return None
    best = None
    for sc in (1e-9, 1e-6, 1e-3, 1.0):
        f = 1.0 / (d * sc)
        if 1 <= f <= 20000:
            err = min(abs(np.log(f / r_)) for r_ in (100., 200., 500.))
            if best is None or err < best[1]:
                best = (f, err)
    return best[0] if best else None


def load_vib(p):
    try:
        df = pd.read_csv(p)
    except Exception:
        return None, None
    df.columns = [str(c).strip().lower() for c in df.columns]
    if not {'x', 'y', 'z'} <= set(df.columns):
        return None, None
    a = df[['x', 'y', 'z']].apply(pd.to_numeric, errors='coerce').dropna().to_numpy(float)
    fs = None
    for tc in TCOLS:
        if tc in df.columns:
            tv = pd.to_numeric(df[tc], errors='coerce').dropna().to_numpy(float)
            if len(tv) > 10:
                fs = detect_fs(tv)
            break
    return (a if len(a) > 100 else None), fs


def adur(p):
    for kw in ('path', 'filename'):
        try:
            return float(librosa.get_duration(**{kw: p}))
        except Exception:
            pass
    return np.nan


def aud_feat(p, offset=0.0, dur=WIN):
    try:
        y, sr = librosa.load(p, sr=16000, mono=True, offset=offset, duration=dur)
    except Exception:
        return None
    if len(y) < 4096:
        return None
    m = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    c = librosa.feature.spectral_centroid(y=y, sr=sr)
    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    ro = librosa.feature.spectral_rolloff(y=y, sr=sr)
    z = librosa.feature.zero_crossing_rate(y)
    return np.concatenate([m.mean(1), m.std(1), c.mean(1), c.std(1),
                           bw.mean(1), bw.std(1), ro.mean(1), z.mean(1)])


def vib_summ(a, fs, secs=None):
    if secs:
        a = a[:int(secs * fs)]
    out = []
    for ax in range(3):
        v = a[:, ax]
        mg = np.abs(np.fft.rfft(v - v.mean()))
        fq = np.fft.rfftfreq(len(v), d=1.0 / fs)
        out += [v.mean(), v.std(), np.sqrt((v ** 2).mean()),
                v.max() - v.min(), fq[mg.argmax()]]
    return np.array(out)


print("\nloading (audio at 3 window positions + vibration)...")
recs = []
for i, (_, r) in enumerate(cat.iterrows()):
    if i % 20 == 0:
        print(f"  {i}/{len(cat)}", flush=True)
    a, fs = load_vib(r['vibr'])
    if a is None or not fs:
        continue
    d = adur(r['audio'])
    if not np.isfinite(d) or d < WIN:
        continue
    f_start = aud_feat(r['audio'], 0.0)
    f_mid = aud_feat(r['audio'], max(0.0, d / 2 - WIN / 2))
    f_end = aud_feat(r['audio'], max(0.0, d - WIN))
    if f_start is None or f_mid is None or f_end is None:
        continue
    recs.append({'label': r['label'], 'printer': r['printer'],
                 'system': 'iPhone' if fs < 350 else 'Teensy',
                 'dur_s': d, 'vib_n': len(a), 'fs': fs,
                 'a_start': f_start, 'a_mid': f_mid, 'a_end': f_end,
                 'v': vib_summ(a, fs, secs=116.9)})

D = pd.DataFrame(recs)
print(f"\nusable: {len(D)}")
A_s = np.nan_to_num(np.vstack(D['a_start']))
A_m = np.nan_to_num(np.vstack(D['a_mid']))
A_e = np.nan_to_num(np.vstack(D['a_end']))
V = np.nan_to_num(np.vstack(D['v']))
DUR = np.c_[D['dur_s'].to_numpy(), D['vib_n'].to_numpy()]
y = LabelEncoder().fit_transform(D['label'])
lab = D['label'].to_numpy()
sysv = D['system'].to_numpy()
prv = D['printer'].to_numpy()
R['n'] = int(len(D))


# ===================== ALT-1  DURATION-STRATIFIED ============================
print("\n" + "=" * 78)
print("ALT-1  DURATION-STRATIFIED CLASSIFICATION")
print("      Within a stratum every class prints for ~the same time, so")
print("      duration carries NO information. Anything above chance here is")
print("      geometry.")
print("=" * 78)

STRATA = {
    'keys_~5min': ['01_key_easy', '02_key_medium', '03_key_hard', '04_key_steps'],
    'large_2to3h': ['06_Autodesk_kickstarter_FDM_test', '07_NIST_additive_test',
                    '08_ASTM', '09_All_in_1'],
}
R['strata'] = {}
for sname, cls in STRATA.items():
    m = np.isin(lab, cls)
    if m.sum() < 8:
        continue
    ys = LabelEncoder().fit_transform(lab[m])
    ch = 100 / len(set(ys))
    dsub = D.loc[m, 'dur_s']
    print(f"\n--- {sname}: {len(cls)} classes, n={int(m.sum())}, "
          f"chance {ch:.2f}% ---")
    print(f"    duration within stratum: {dsub.min():.0f}-{dsub.max():.0f} s "
          f"(ratio {dsub.max()/dsub.min():.2f}x)")
    ent = {}
    for nm, X in (('duration only', DUR[m]), ('vibration', V[m]),
                  ('acoustic 60s start', A_s[m]), ('acoustic 60s mid', A_m[m]),
                  ('vib + acoustic', np.hstack([V, A_m])[m])):
        a_, _ = cvacc(X, ys)
        if a_ is None:
            continue
        ci = wilson(a_, int(m.sum()))
        sig = "ABOVE chance" if ci[0] > ch else "n.s."
        ent[nm] = {'acc': a_, 'ci': ci, 'sig': sig}
        print(f"    {nm:22s} {a_:6.2f}%  CI {ci}   {sig}")
    # permutation null for the best non-duration channel
    best = max((k for k in ent if k != 'duration only'),
               key=lambda k: ent[k]['acc'], default=None)
    if best:
        Xb = {'vibration': V, 'acoustic 60s start': A_s,
              'acoustic 60s mid': A_m,
              'vib + acoustic': np.hstack([V, A_m])}[best][m]
        ent[best]['perm_p'] = perm_p(Xb, ys, ent[best]['acc'], tag=sname)
        print(f"    permutation p for {best}: {ent[best]['perm_p']}")
    ent['chance'] = round(ch, 2)
    ent['n'] = int(m.sum())
    ent['duration_ratio'] = round(float(dsub.max() / dsub.min()), 3)
    R['strata'][sname] = ent

print("\n  INTERPRETATION: any channel with a CI lower bound above chance in a")
print("  stratum demonstrates geometry-dependent leakage that duration cannot")
print("  explain. That isolates geometry from elapsed time.")


# ===================== ALT-2  ACOUSTIC WINDOW POSITION =======================
print("\n" + "=" * 78)
print("ALT-2  ACOUSTIC WINDOW POSITION — startup signature or steady state?")
print("=" * 78)
ch12 = 100 / len(set(y))
R['acoustic_position'] = {}
for nm, X in (('start 60s', A_s), ('middle 60s', A_m), ('end 60s', A_e)):
    a_, _ = cvacc(X, y)
    ci = wilson(a_, len(y))
    R['acoustic_position'][nm] = {'acc': a_, 'ci': ci}
    print(f"  {nm:12s} {a_:6.2f}%  CI {ci}")
print(f"  chance {ch12:.2f}%")
print("\n  If START is high and MIDDLE/END are at chance, the acoustic result")
print("  is a startup/first-layer signature and AMNC still suppresses the")
print("  running-motor channel — the original claim survives, refined.")
print("  If all three are elevated, acoustic leakage is steady-state.")


# ===================== ALT-3  ACOUSTIC PER CAPTURE SYSTEM ====================
print("\n" + "=" * 78)
print("ALT-3  ACOUSTIC PER CAPTURE SYSTEM (different microphones)")
print("=" * 78)
R['acoustic_by_system'] = {}
for s in sorted(set(sysv)):
    m = sysv == s
    d = {}
    for nm, X in (('start', A_s), ('mid', A_m)):
        a_, _ = cvacc(X[m], y[m])
        d[nm] = {'acc': a_, 'ci': wilson(a_, int(m.sum()))}
    R['acoustic_by_system'][s] = d
    print(f"  {s:8s} n={int(m.sum()):3d}  start {d['start']['acc']:6.2f}% "
          f"CI {d['start']['ci']}  |  mid {d['mid']['acc']:6.2f}% "
          f"CI {d['mid']['ci']}")
print("\n  Elevated in only ONE system => capture artifact, not a channel.")


# ===================== ALT-4  DOES ACOUSTIC ADD OVER DURATION? ===============
print("\n" + "=" * 78)
print("ALT-4  DOES ACOUSTIC ADD INFORMATION OVER DURATION?")
print("=" * 78)
a_d, p_d = cvacc(DUR, y)
a_da, p_da = cvacc(np.hstack([DUR, A_m]), y)
a_dv, p_dv = cvacc(np.hstack([DUR, V]), y)
print(f"  duration only        {a_d:6.2f}%")
print(f"  duration + acoustic  {a_da:6.2f}%   ({a_da - a_d:+.2f})")
print(f"  duration + vibration {a_dv:6.2f}%   ({a_dv - a_d:+.2f})")
R['dur_only'] = a_d
R['dur_plus_acoustic'] = a_da
R['dur_plus_vib'] = a_dv
for nm, pp in (('acoustic', p_da), ('vibration', p_dv)):
    cd, cb = (p_d == y).astype(int), (pp == y).astype(int)
    tb = [[int(((cd == 1) & (cb == 1)).sum()), int(((cd == 1) & (cb == 0)).sum())],
          [int(((cd == 0) & (cb == 1)).sum()), int(((cd == 0) & (cb == 0)).sum())]]
    pv = round(float(mcnemar(tb, exact=True).pvalue), 6)
    R[f'mcnemar_{nm}'] = {'p': pv, 'table': tb}
    print(f"  McNemar duration vs duration+{nm}: p = {pv}  {tb}")


# ===================== ALT-5  SPECTRAL EVIDENCE OF SUPPRESSION ===============
print("\n" + "=" * 78)
print("ALT-5  SPECTRAL EVIDENCE THAT AMNC IS ACTING")
print("      Motor-resonance band 120-360 Hz vs the dataset's own background")
print("      Noise Recordings. Independent of any classifier.")
print("=" * 78)
noise = []
for p in BASE.rglob('*'):
    if p.is_file() and p.suffix.lower() in AUD and 'Noise Recordings' in p.parts:
        noise.append(p)
print(f"background noise recordings found: {len(noise)}")


def mean_spec(paths, n=12, offset=0.0):
    S = []
    for p in paths[:n]:
        try:
            yy, sr = librosa.load(str(p), sr=16000, mono=True,
                                  offset=offset, duration=30)
        except Exception:
            continue
        if len(yy) < 4096:
            continue
        f, Pxx = None, None
        from scipy.signal import welch
        f, Pxx = welch(yy, fs=sr, nperseg=4096)
        S.append(Pxx)
    return (f, np.mean(S, axis=0)) if S else (None, None)


pr_paths = [Path(p) for p in D['audio'].tolist()] if 'audio' in D else \
           [Path(p) for p in cat['audio'].tolist()]
f_pr, S_pr = mean_spec(pr_paths, n=24, offset=30.0)
f_no, S_no = mean_spec(noise, n=24) if noise else (None, None)

if S_pr is not None:
    band = (f_pr >= 120) & (f_pr <= 360)
    ref = (f_pr >= 600) & (f_pr <= 2000)
    ratio_pr = float(10 * np.log10(S_pr[band].mean() / S_pr[ref].mean()))
    R['print_band_ratio_db'] = round(ratio_pr, 2)
    print(f"  print audio: motor band (120-360 Hz) relative to 600-2000 Hz "
          f"reference = {ratio_pr:+.2f} dB")
    if S_no is not None:
        ratio_no = float(10 * np.log10(S_no[band].mean() / S_no[ref].mean()))
        R['noise_band_ratio_db'] = round(ratio_no, 2)
        R['band_suppression_db'] = round(ratio_pr - ratio_no, 2)
        print(f"  background : same ratio = {ratio_no:+.2f} dB")
        print(f"  difference : {ratio_pr - ratio_no:+.2f} dB")
        print("\n  A SMALL difference means the motor band barely rises above")
        print("  background during printing — direct spectral evidence that the")
        print("  motor acoustic signature is suppressed, independent of any")
        print("  classifier, and without needing an off-switch the hardware does not expose.")

    fig, ax = plt.subplots(figsize=(7.5, 4.4))
    ax.semilogy(f_pr, S_pr, lw=1.1, color='#0072B2', label='During printing')
    if S_no is not None:
        ax.semilogy(f_no, S_no, lw=1.1, color='#999999', label='Background noise')
    ax.axvspan(120, 360, color='#E69F00', alpha=0.25,
               label='Motor resonance band')
    ax.set_xlim(0, 3000)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD')
    ax.legend()
    fig.tight_layout()
    for e in ('png', 'pdf'):
        fig.savefig(f'{OUT}/fig13_spectrum.{e}')
    plt.close(fig)
    print(f"  saved fig13_spectrum")


# ================================ OUTPUT =====================================
R['runtime_min'] = round((time.time() - t0) / 60, 1)
Path(f'{OUT}/results_v6.json').write_text(json.dumps(R, indent=2, default=str))
print("\n" + "=" * 78)
print(f"DONE — {R['runtime_min']} min  ->  {OUT}")
print("=" * 78)
print(json.dumps(R, indent=2, default=str))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
catalog: 144 recordings | 12 classes

loading (audio at 3 window positions + vibration)...
  0/144
  20/144
  40/144
  60/144
  80/144
  100/144
  120/144
  140/144

usable: 144

ALT-1  DURATION-STRATIFIED CLASSIFICATION
      Within a stratum every class prints for ~the same time, so
      duration carries NO information. Anything above chance here is
      geometry.

--- keys_~5min: 4 classes, n=48, chance 25.00% ---
    duration within stratum: 88-543 s (ratio 6.15x)
    duration only           14.58%  CI (7.25, 27.17)   n.s.
    vibration               22.92%  CI (13.31, 36.54)   n.s.
    acoustic 60s start      16.67%  CI (8.7, 29.58)   n.s.
    acoustic 60s mid        18.75%  CI (10.19, 31.94)   n.s.
    vib + acoustic          18.75%  CI (10.19, 31.94)   n.s.
      perm keys_~5min 0/1000
      perm keys_~5min 200/1000
      perm keys_~5min 400/1000
   

---
## Stage 5 · Duration controls

Because duration is the strongest discriminator, every other channel is assessed
against it rather than against chance.

- **T1 incremental contribution** — within the long-duration stratum duration alone
  reaches 95.83%; adding sensor features never corrects a duration-only error
  (McNemar bottom-left cell is zero for every combination).
- **Fixed-offset acoustic window** — 60–120 s, identical for every recording:
  31.25%, CI [19.95, 45.33] against a 25% baseline, interval includes chance.
- **T2 equalised observation window** — truncation to a uniform 543 s:
  **45.83%**, permutation *p* = 0.023. Cleanest duration-equalised evidence.
- **T3 duration-overlap subset** — **inconclusive**: duration alone still classifies
  the resulting subset at 60.34%, so binning did not remove duration separation.
  Reported for completeness and not relied upon.


In [ ]:
# =============================================================================
#  STAGE 5 — DURATION CONTROLS (T1, T2, T3)
#
#  T1  within-stratum McNemar: does anything add information over duration alone?
#  T2  truncation to a uniform observation window
#  T3  duration-overlap subset via quantile binning  (CONTROL FAILED - see output)
#  Plus the fixed 60-120 s offset acoustic window.
#
#  Each test prints its own duration-only control. If that control is not near
#  chance, the test did not control duration and its result cannot be read as
#  geometry — T3 is exactly this case and is reported as inconclusive.
# =============================================================================

!pip install -q librosa scikit-learn statsmodels

import os, re, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import librosa
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar

warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# ============================== CONFIG =======================================
DRIVE_BASE = '/content/drive/MyDrive/3dprinter_sidechannel'
OUT        = '/content/drive/MyDrive/3dprinter_sidechannel/revision_v7'
WIN        = 60            # acoustic window (s)
TRUNC_S    = 543.0         # T2: keys-stratum maximum duration
B_PERM     = 1000
RANDOM_STATE = 42
LARGE = ['06_Autodesk_kickstarter_FDM_test', '07_NIST_additive_test',
         '08_ASTM', '09_All_in_1']
KEYS  = ['01_key_easy', '02_key_medium', '03_key_hard', '04_key_steps']
# =============================================================================

os.makedirs(OUT, exist_ok=True)
np.random.seed(RANDOM_STATE)
t0 = time.time()
R = {}
BASE = Path(DRIVE_BASE)
SKIP = {'results', 'results_v2', 'revision_v3', 'revision_v4', 'revision_v5',
        'revision_v6', 'revision_v7', 'output', 'Printing Videos',
        'Noise Recordings', 'Artifacts', '__MACOSX', 'figures'}
AUD = {'.mp3', '.caf', '.wav', '.m4a'}

plt.rcParams.update({'font.size': 12, 'font.family': 'serif',
                     'figure.dpi': 300, 'savefig.dpi': 300,
                     'savefig.bbox': 'tight', 'axes.grid': True,
                     'grid.alpha': 0.3, 'grid.linestyle': '--'})


def wilson(a, n):
    if not n:
        return (float('nan'),) * 2
    lo, hi = proportion_confint(int(round(a / 100 * n)), n, 0.05, 'wilson')
    return round(lo * 100, 2), round(hi * 100, 2)


def rf():
    return Pipeline([('s', StandardScaler()),
                     ('m', RandomForestClassifier(200, random_state=RANDOM_STATE,
                                                  n_jobs=-1))])


def cvpred(X, yy, folds=5):
    n_min = pd.Series(yy).value_counts().min()
    k = int(min(folds, n_min))
    if k < 2 or len(set(yy)) < 2:
        return None, None
    p = cross_val_predict(rf(), np.nan_to_num(X), yy,
                          cv=StratifiedKFold(k, shuffle=True,
                                             random_state=RANDOM_STATE))
    return round(accuracy_score(yy, p) * 100, 2), p


def perm_p(X, yy, obs, B=B_PERM, tag=""):
    rng, r = np.random.default_rng(RANDOM_STATE), 0
    for b in range(B):
        if b % max(1, B // 4) == 0:
            print(f"        perm {tag} {b}/{B}", flush=True)
        a, _ = cvpred(X, rng.permutation(yy))
        if a is not None and a >= obs:
            r += 1
    return round((r + 1) / (B + 1), 5)


def mc(p_base, p_test, yy):
    cb, ct = (p_base == yy).astype(int), (p_test == yy).astype(int)
    tb = [[int(((cb == 1) & (ct == 1)).sum()), int(((cb == 1) & (ct == 0)).sum())],
          [int(((cb == 0) & (ct == 1)).sum()), int(((cb == 0) & (ct == 0)).sum())]]
    return round(float(mcnemar(tb, exact=True).pvalue), 6), tb


# =============================== LOAD ========================================
rows = []
for obj in sorted(BASE.iterdir()):
    if not obj.is_dir() or obj.name in SKIP or not re.match(r'^\d\d_', obj.name):
        continue
    nested = sorted([d for d in obj.iterdir() if d.is_dir()])
    if not nested:
        continue
    for pr in sorted(nested[0].iterdir()):
        if not pr.is_dir() or pr.name in SKIP:
            continue
        sess = {}
        for f in sorted(pr.rglob('*')):
            e = f.suffix.lower()
            if e in AUD or e == '.csv':
                s = sess.setdefault(f.parent, {'a': [], 'c': []})
                (s['a'] if e in AUD else s['c']).append(f)
        for folder in sorted(sess):
            fs_ = sess[folder]
            if fs_['a'] and fs_['c']:
                for af in sorted(fs_['a']):
                    rows.append({'label': obj.name, 'printer': pr.name,
                                 'audio': str(af), 'vibr': str(sorted(fs_['c'])[0])})
cat = (pd.DataFrame(rows).sort_values(['label', 'printer', 'audio'])
         .reset_index(drop=True))
assert len(cat) == 144 and cat['label'].nunique() == 12, f"CATALOG WRONG {len(cat)}"
print(f"catalog OK: {len(cat)} / {cat['label'].nunique()} classes")

TCOLS = ('t', 'time', 'timestamp', 'millis', 'micros', 'ms', 'us', 'elapsed')


def detect_fs(tv):
    d = np.median(np.diff(tv))
    if not np.isfinite(d) or d <= 0:
        return None
    best = None
    for sc in (1e-9, 1e-6, 1e-3, 1.0):
        f = 1.0 / (d * sc)
        if 1 <= f <= 20000:
            err = min(abs(np.log(f / r_)) for r_ in (100., 200., 500.))
            if best is None or err < best[1]:
                best = (f, err)
    return best[0] if best else None


def load_vib(p):
    try:
        df = pd.read_csv(p)
    except Exception:
        return None, None
    df.columns = [str(c).strip().lower() for c in df.columns]
    if not {'x', 'y', 'z'} <= set(df.columns):
        return None, None
    a = df[['x', 'y', 'z']].apply(pd.to_numeric, errors='coerce').dropna().to_numpy(float)
    fs = None
    for tc in TCOLS:
        if tc in df.columns:
            tv = pd.to_numeric(df[tc], errors='coerce').dropna().to_numpy(float)
            if len(tv) > 10:
                fs = detect_fs(tv)
            break
    return (a if len(a) > 100 else None), fs


def adur(p):
    for kw in ('path', 'filename'):
        try:
            return float(librosa.get_duration(**{kw: p}))
        except Exception:
            pass
    return np.nan


def aud_feat(p, offset=0.0, dur=WIN):
    try:
        yy, sr = librosa.load(p, sr=16000, mono=True, offset=offset, duration=dur)
    except Exception:
        return None
    if len(yy) < 4096:
        return None
    m = librosa.feature.mfcc(y=yy, sr=sr, n_mfcc=13)
    c = librosa.feature.spectral_centroid(y=yy, sr=sr)
    bw = librosa.feature.spectral_bandwidth(y=yy, sr=sr)
    ro = librosa.feature.spectral_rolloff(y=yy, sr=sr)
    z = librosa.feature.zero_crossing_rate(yy)
    return np.concatenate([m.mean(1), m.std(1), c.mean(1), c.std(1),
                           bw.mean(1), bw.std(1), ro.mean(1), z.mean(1)])


def vib_summ(a, fs, secs=None):
    if secs:
        a = a[:max(200, int(secs * fs))]
    out = []
    for ax in range(3):
        v = a[:, ax]
        mg = np.abs(np.fft.rfft(v - v.mean()))
        fq = np.fft.rfftfreq(len(v), d=1.0 / fs)
        out += [v.mean(), v.std(), np.sqrt((v ** 2).mean()),
                v.max() - v.min(), fq[mg.argmax()]]
    return np.array(out)


print("\nloading...")
recs = []
for i, (_, r) in enumerate(cat.iterrows()):
    if i % 24 == 0:
        print(f"  {i}/{len(cat)}", flush=True)
    a, fs = load_vib(r['vibr'])
    if a is None or not fs:
        continue
    d = adur(r['audio'])
    if not np.isfinite(d) or d < WIN:
        continue
    # acoustic: start window, and a window at a FIXED absolute offset so that
    # window position does not covary with total duration
    f_start = aud_feat(r['audio'], 0.0)
    f_fixed = aud_feat(r['audio'], 60.0)          # 60-120 s for every recording
    if f_start is None or f_fixed is None:
        continue
    recs.append({'label': r['label'], 'printer': r['printer'],
                 'dur_s': d, 'vib_n': len(a), 'fs': fs,
                 'vib_full': vib_summ(a, fs),
                 'vib_trunc': vib_summ(a, fs, secs=TRUNC_S),
                 'a_start': f_start, 'a_fixed': f_fixed})

D = pd.DataFrame(recs)
print(f"usable {len(D)}")
R['n_total'] = int(len(D))


def blocks(sub):
    """Return feature blocks for a subset dataframe."""
    return dict(
        dur=np.nan_to_num(np.c_[sub['dur_s'].to_numpy(), sub['vib_n'].to_numpy()]),
        vib=np.nan_to_num(np.vstack(sub['vib_full'])),
        vib_trunc=np.nan_to_num(np.vstack(sub['vib_trunc'])),
        aco=np.nan_to_num(np.vstack(sub['a_start'])),
        aco_fixed=np.nan_to_num(np.vstack(sub['a_fixed'])),
    )


# ================================ T1 =========================================
print("\n" + "=" * 78)
print("T1  WITHIN-STRATUM McNEMAR — does anything beat duration inside")
print("    large_2to3h, where duration alone scored 95.83%?")
print("=" * 78)
sub = D[D['label'].isin(LARGE)].reset_index(drop=True)
ys = LabelEncoder().fit_transform(sub['label'])
n = len(sub)
ch = 100 / len(set(ys))
B = blocks(sub)
print(f"n={n} | 4 classes | chance {ch:.2f}%")
print(f"duration range {sub['dur_s'].min():.0f}-{sub['dur_s'].max():.0f} s")
print("\nper-class duration (s):")
print(sub.groupby('label')['dur_s'].agg(['min', 'median', 'max']).round(0).to_string())

# does duration separate the classes? (tests Gemini's argument-1 premise)
ov = sub.groupby('label')['dur_s'].agg(['min', 'max'])
pairs_overlap = 0
labs_ = list(ov.index)
for i in range(len(labs_)):
    for j in range(i + 1, len(labs_)):
        a1, b1 = ov.loc[labs_[i], 'min'], ov.loc[labs_[i], 'max']
        a2, b2 = ov.loc[labs_[j], 'min'], ov.loc[labs_[j], 'max']
        if max(a1, a2) <= min(b1, b2):
            pairs_overlap += 1
R['T1_class_duration_overlapping_pairs'] = pairs_overlap
R['T1_class_pairs_total'] = len(labs_) * (len(labs_) - 1) // 2
print(f"\nclass duration ranges overlapping: {pairs_overlap} of "
      f"{R['T1_class_pairs_total']} pairs")
print("  (Gemini argument 1 requires substantial overlap here)")

a_dur, p_dur = cvpred(B['dur'], ys)
print(f"\n  duration only                 {a_dur:6.2f}%  CI {wilson(a_dur, n)}")
R['T1'] = {'n': n, 'chance': round(ch, 2), 'duration_only': a_dur,
           'duration_only_ci': wilson(a_dur, n), 'tests': {}}

combos = {
    'duration + vibration':            np.hstack([B['dur'], B['vib']]),
    'duration + acoustic':             np.hstack([B['dur'], B['aco']]),
    'duration + vib + acoustic':       np.hstack([B['dur'], B['vib'], B['aco']]),
    'vibration alone':                 B['vib'],
    'acoustic alone (start)':          B['aco'],
    'acoustic alone (fixed 60-120s)':  B['aco_fixed'],
    'vib + acoustic alone':            np.hstack([B['vib'], B['aco']]),
}
for nm, X in combos.items():
    a_, p_ = cvpred(X, ys)
    if a_ is None:
        continue
    ent = {'acc': a_, 'ci': wilson(a_, n), 'delta_vs_duration': round(a_ - a_dur, 2)}
    if nm.startswith('duration +'):
        pv, tb = mc(p_dur, p_, ys)
        ent['mcnemar_p'] = pv
        ent['mcnemar_table'] = tb
        verdict = ("ADDS over duration" if (pv < 0.05 and a_ > a_dur)
                   else "no significant gain")
        print(f"  {nm:31s} {a_:6.2f}%  CI {ent['ci']}  "
              f"delta {ent['delta_vs_duration']:+6.2f}  McNemar p={pv}  {verdict}")
    else:
        print(f"  {nm:31s} {a_:6.2f}%  CI {ent['ci']}")
    R['T1']['tests'][nm] = ent

best_add = max((k for k in R['T1']['tests'] if k.startswith('duration +')),
               key=lambda k: R['T1']['tests'][k]['acc'])
R['T1']['verdict'] = (
    "SENSORS ADD OVER DURATION" if any(
        R['T1']['tests'][k].get('mcnemar_p', 1) < 0.05 and
        R['T1']['tests'][k]['delta_vs_duration'] > 0
        for k in R['T1']['tests'] if k.startswith('duration +'))
    else "NO SENSOR GAIN OVER DURATION")
print(f"\n  T1 VERDICT: {R['T1']['verdict']}")
print("  If NO GAIN: the 72.92% stratum result is explained by duration.")
print("  If ADDS:    geometry-dependent leakage is demonstrated.")


# ================================ T2 =========================================
print("\n" + "=" * 78)
print(f"T2  TRUNCATION CONTROL — truncate large_2to3h vibration to {TRUNC_S:.0f} s")
print("    (the keys-stratum maximum) and re-test")
print("=" * 78)
a_vt, _ = cvpred(B['vib_trunc'], ys)
a_vt_a, p_vt_a = cvpred(np.hstack([B['vib_trunc'], B['aco']]), ys)
print(f"  vibration, full recording        {R['T1']['tests']['vibration alone']['acc']:6.2f}%")
print(f"  vibration, truncated to {TRUNC_S:.0f}s     {a_vt:6.2f}%  CI {wilson(a_vt, n)}")
print(f"  vib(trunc) + acoustic            {a_vt_a:6.2f}%  CI {wilson(a_vt_a, n)}")
pp = perm_p(np.hstack([B['vib_trunc'], B['aco']]), ys, a_vt_a, tag="T2")
print(f"  permutation p = {pp}")
R['T2'] = {'trunc_s': TRUNC_S, 'vib_trunc': a_vt, 'vib_trunc_ci': wilson(a_vt, n),
           'vib_trunc_plus_acoustic': a_vt_a, 'perm_p': pp,
           'vib_full': R['T1']['tests']['vibration alone']['acc']}
R['T2']['verdict'] = ("SURVIVES truncation -> runtime-threshold explanation FAILS"
                      if wilson(a_vt_a, n)[0] > ch else
                      "COLLAPSES under truncation -> threshold explanation SUPPORTED")
print(f"\n  T2 VERDICT: {R['T2']['verdict']}")


# ================================ T3 =========================================
print("\n" + "=" * 78)
print("T3  DURATION-OVERLAP SUBSET — the test Gemini's argument 1 assumes exists")
print("=" * 78)
# Build a subset in which duration genuinely cannot separate classes: bin all
# 144 recordings by duration, keep only bins containing >=2 classes, and within
# each bin keep a balanced sample.
D2 = D.copy()
D2['bin'] = pd.qcut(D2['dur_s'], q=12, duplicates='drop')
keep_idx = []
bin_report = []
for b, g in D2.groupby('bin', observed=True):
    ncl = g['label'].nunique()
    if ncl >= 2:
        m = g['label'].value_counts().min()
        take = g.groupby('label', group_keys=False).apply(
            lambda x: x.sample(n=min(len(x), m), random_state=RANDOM_STATE))
        keep_idx += list(take.index)
        bin_report.append({'bin': str(b), 'classes': ncl, 'kept': len(take),
                           'dur_min': round(g['dur_s'].min(), 1),
                           'dur_max': round(g['dur_s'].max(), 1)})
sub3 = D2.loc[sorted(set(keep_idx))].reset_index(drop=True)
print(f"bins with >=2 classes: {len(bin_report)}")
for br in bin_report:
    print(f"   dur {br['dur_min']:8.1f}-{br['dur_max']:8.1f} s | "
          f"{br['classes']} classes | n={br['kept']}")

R['T3'] = {'n': int(len(sub3)), 'bins': bin_report}
if len(sub3) >= 16 and sub3['label'].nunique() >= 2:
    y3 = LabelEncoder().fit_transform(sub3['label'])
    n3 = len(sub3)
    ch3 = 100 / len(set(y3))
    B3 = blocks(sub3)
    print(f"\nsubset n={n3} | {len(set(y3))} classes | chance {ch3:.2f}%")
    print(f"duration range {sub3['dur_s'].min():.1f}-{sub3['dur_s'].max():.1f} s")
    R['T3'].update(classes=int(len(set(y3))), chance=round(ch3, 2))
    for nm, X in (('duration only (CONTROL)', B3['dur']),
                  ('vibration', B3['vib']),
                  ('acoustic (fixed 60-120s)', B3['aco_fixed']),
                  ('vib + acoustic', np.hstack([B3['vib'], B3['aco_fixed']]))):
        a_, _ = cvpred(X, y3)
        if a_ is None:
            continue
        ci = wilson(a_, n3)
        sig = "ABOVE chance" if ci[0] > ch3 else "n.s."
        R['T3'][nm] = {'acc': a_, 'ci': ci, 'sig': sig}
        print(f"  {nm:26s} {a_:6.2f}%  CI {ci}   {sig}")
    if 'vib + acoustic' in R['T3']:
        R['T3']['perm_p_vib_acoustic'] = perm_p(
            np.hstack([B3['vib'], B3['aco_fixed']]), y3,
            R['T3']['vib + acoustic']['acc'], tag="T3")
        print(f"  permutation p (vib+acoustic) = {R['T3']['perm_p_vib_acoustic']}")
    dctrl = R['T3'].get('duration only (CONTROL)', {}).get('acc', 999)
    R['T3']['control_valid'] = bool(dctrl <= ch3 * 1.6)
    print(f"\n  CONTROL CHECK: duration-only = {dctrl}% vs chance {ch3:.2f}%")
    print("  " + ("control VALID — duration cannot separate this subset"
                  if R['T3']['control_valid'] else
                  "control INVALID — duration still separates; result not geometry"))
else:
    print("\ninsufficient overlap to build a duration-matched subset.")
    print("If no duration bin contains 2+ classes, class durations are DISJOINT,")
    print("which itself refutes the premise that classes have overlapping times.")
    R['T3']['verdict'] = 'NO_OVERLAP_EXISTS'


# ============================== OUTPUT =======================================
fig, ax = plt.subplots(figsize=(9, 4.6))
names, vals, cols = [], [], []
names.append('T1 duration\nonly'); vals.append(R['T1']['duration_only']); cols.append('#D55E00')
for k in ('vibration alone', 'vib + acoustic alone', 'duration + vib + acoustic'):
    if k in R['T1']['tests']:
        names.append('T1 ' + k.replace(' ', '\n')); vals.append(R['T1']['tests'][k]['acc'])
        cols.append('#0072B2')
names.append(f'T2 vib trunc\n+acoustic'); vals.append(R['T2']['vib_trunc_plus_acoustic'])
cols.append('#009E73')
if 'vib + acoustic' in R.get('T3', {}):
    names.append('T3 overlap\nvib+acoustic'); vals.append(R['T3']['vib + acoustic']['acc'])
    cols.append('#CC79A7')
b = ax.bar(names, vals, color=cols, edgecolor='black')
for bb, vv in zip(b, vals):
    ax.text(bb.get_x() + bb.get_width() / 2, vv + 1, f'{vv:.1f}%', ha='center', fontsize=9)
ax.axhline(25, color='red', ls='--', label='Chance (4-class, 25%)')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, max(vals) + 12)
ax.legend()
plt.xticks(fontsize=8)
fig.tight_layout()
for e in ('png', 'pdf'):
    fig.savefig(f'{OUT}/fig14_three_tests.{e}')
plt.close(fig)

R['runtime_min'] = round((time.time() - t0) / 60, 1)
Path(f'{OUT}/results_v7.json').write_text(json.dumps(R, indent=2, default=str))

print("\n" + "=" * 78)
print("SUMMARY")
print("=" * 78)
print(f"T1  {R['T1']['verdict']}")
print(f"T2  {R['T2']['verdict']}")
print(f"T3  {R['T3'].get('verdict', 'control_valid=' + str(R['T3'].get('control_valid')))}")
print(f"\nDONE — {R['runtime_min']} min  ->  {OUT}")
print(json.dumps(R, indent=2, default=str))

Mounted at /content/drive
catalog OK: 144 / 12 classes

loading...
  0/144
  24/144
  48/144
  72/144
  96/144
  120/144
usable 144

T1  WITHIN-STRATUM McNEMAR — does anything beat duration inside
    large_2to3h, where duration alone scored 95.83%?
n=48 | 4 classes | chance 25.00%
duration range 2089-11273 s

per-class duration (s):
                                     min  median      max
label                                                    
06_Autodesk_kickstarter_FDM_test  2489.0  5859.0   9064.0
07_NIST_additive_test             2089.0  4836.0   7982.0
08_ASTM                           3086.0  7072.0  11273.0
09_All_in_1                       2774.0  6362.0  10190.0

class duration ranges overlapping: 6 of 6 pairs
  (Gemini argument 1 requires substantial overlap here)

  duration only                  95.83%  CI (86.02, 98.85)
  duration + vibration             79.17%  CI (65.74, 88.27)  delta -16.66  McNemar p=0.007812  no significant gain
  duration + acoustic              

---
## Stage 6 · Summary figures

Single pass producing the five summary figures, reading values from
the JSON written by the stages above so text and figures cannot diverge. Every plotted
value is echoed to the console and written to `figure_values.json`.

Retired figures from earlier drafts are not regenerated; the cell warns if any are still
present in the output folder.


In [1]:
# =============================================================================
#  STAGE 6 — SUMMARY FIGURES
#
#     fig1_acoustic_window.png     observation-window sweep + truncation audit
#     fig2_gcode_validation.png    recording duration vs G-code print time
#     fig3_amplitude_vs_shape.png  vibration feature ablation (hertz)
#     fig4_per_device.png          per-printer accuracy + cross-printer transfer
#     fig5_spectrum.png            motor band vs background PSD
#
#  Values are read from the archived result JSON where they exist; only the G-code
#  scatter and the spectrum touch raw data.
# =============================================================================

!pip install -q librosa matplotlib pandas numpy scipy

import os, json, glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# ============================== CONFIG =======================================
DRIVE_BASE = '/content/drive/MyDrive/3dprinter_sidechannel'
OUT        = f'{DRIVE_BASE}/figures'
V3   = f'{DRIVE_BASE}/revision_v3/results.json'
V5   = f'{DRIVE_BASE}/revision_v5/results_v5.json'
V6   = f'{DRIVE_BASE}/revision_v6/results_v6.json'
META = f'{DRIVE_BASE}/revision_v5/recording_metadata.csv'
GCSV = f'{DRIVE_BASE}/revision_v5/gcode_times.csv'
CHANCE = 8.33
# =============================================================================

os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({
    'font.size': 11, 'font.family': 'serif',
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--',
    'axes.axisbelow': True,
})

BLUE, GREEN, ORANGE, GREY, RED, PURPLE = (
    '#0072B2', '#009E73', '#E69F00', '#999999', '#D55E00', '#CC79A7')


def load(p):
    try:
        return json.loads(Path(p).read_text())
    except Exception as e:
        print(f"  ! could not read {p}: {e}")
        return {}


r3, r5, r6 = load(V3), load(V5), load(V6)
LOG = {}


def save(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(f'{OUT}/{name}.{ext}')
    plt.close(fig)
    print(f"  saved {name}")


def bars(ax, labels, vals, colors, chance=None, fmt='{:.2f}%', pad=1.0):
    b = ax.bar(labels, vals, color=colors, edgecolor='black', linewidth=0.7)
    top = max(v for v in vals if np.isfinite(v))
    for bb, v in zip(b, vals):
        if np.isfinite(v):
            ax.text(bb.get_x() + bb.get_width() / 2, v + pad, fmt.format(v),
                    ha='center', fontsize=9)
    if chance is not None:
        ax.axhline(chance, color='red', ls='--', lw=1.2,
                   label=f'Chance ({chance:.2f}%)')
    ax.set_ylabel('Classification accuracy (\\%)' if False else 'Classification accuracy (%)')
    ax.set_ylim(0, top + 12)
    return b


# =============================================================================
# FIG 1 — acoustic observation window, with truncation audit
# =============================================================================
print("\nfig1_acoustic_window")
sweep = r3.get('acoustic_window_sweep', {})
multi = r3.get('acoustic_multiwindow', np.nan)
# recordings shorter than each window (audit); recompute if metadata present
trunc = {'10': 0, '30': 0, '60': 0, '120': 20, '300': 36}
if Path(META).exists():
    M = pd.read_csv(META)
    if 'dur_s' in M:
        for w in ('10', '30', '60', '120', '300'):
            trunc[w] = int((M['dur_s'] < float(w)).sum())

order = ['10', '30', '60', '120', '300']
vals = [sweep.get(w, np.nan) for w in order] + [multi]
labs = [f'{w} s' for w in order] + ['8 x 30 s\ndistributed']
# clean windows get solid fill; truncating windows get hatched
cols, hatches = [], []
for w in order:
    clean = trunc.get(w, 0) == 0
    cols.append(BLUE if clean else GREY)
    hatches.append('' if clean else '//')
cols.append(PURPLE); hatches.append('')

fig, ax = plt.subplots(figsize=(7.2, 4.2))
b = ax.bar(labs, vals, color=cols, edgecolor='black', linewidth=0.7)
for bb, h in zip(b, hatches):
    bb.set_hatch(h)
for bb, v, w in zip(b, vals, order + ['mw']):
    if np.isfinite(v):
        ax.text(bb.get_x() + bb.get_width() / 2, v + 1, f'{v:.2f}%',
                ha='center', fontsize=9)
        n = trunc.get(w)
        if n:
            ax.text(bb.get_x() + bb.get_width() / 2, 1.5, f'{n} trunc.',
                    ha='center', fontsize=7.5, color='black')
ax.axhline(CHANCE, color='red', ls='--', lw=1.2, label=f'Chance ({CHANCE}%)')
ax.set_ylabel('Classification accuracy (%)')
ax.set_xlabel('Audio observed per recording')
ax.set_ylim(0, max(v for v in vals if np.isfinite(v)) + 14)
ax.legend(handles=[
    Patch(facecolor=BLUE, edgecolor='black', label='Duration-clean'),
    Patch(facecolor=GREY, edgecolor='black', hatch='//', label='Truncates recordings'),
    Patch(facecolor=PURPLE, edgecolor='black', label='Distributed sampling'),
    plt.Line2D([0], [0], color='red', ls='--', label=f'Chance ({CHANCE}%)'),
], fontsize=8, loc='upper left')
save(fig, 'fig1_acoustic_window')
LOG['fig1'] = dict(zip(labs, vals))
print("   ", LOG['fig1'])

# =============================================================================
# FIG 2 — recording duration vs G-code print time
# =============================================================================
print("\nfig2_gcode_validation")
r_pearson = r5.get('gcode_validation', {}).get('corr_gcode_s_pearson', np.nan)
made = False
if Path(META).exists():
    M = pd.read_csv(META)
    gmap = {}
    if Path(GCSV).exists():
        G = pd.read_csv(GCSV)
        G = G.dropna(subset=['model_s'])
        gmap = G.groupby('label')['model_s'].median().to_dict()
    elif 'gcode_model_times_s' in r5:
        gmap = {k: float(v) for k, v in r5['gcode_model_times_s'].items()}
    if gmap and 'label' in M:
        M['gcode_s'] = M['label'].map(gmap)
        ok = M['gcode_s'].notna() & M['dur_s'].notna()
        if ok.sum() > 10:
            x = M.loc[ok, 'gcode_s'] / 60.0
            y = M.loc[ok, 'dur_s'] / 60.0
            if not np.isfinite(r_pearson):
                r_pearson = float(np.corrcoef(x, y)[0, 1])
            fig, ax = plt.subplots(figsize=(5.4, 5.0))
            for pr, mk, c in (('Bambu_P1P', 'o', BLUE),
                              ('Bambu_A1mini', '^', ORANGE)):
                m = ok & (M['printer'] == pr)
                if m.any():
                    ax.scatter(M.loc[m, 'gcode_s'] / 60, M.loc[m, 'dur_s'] / 60,
                               s=30, marker=mk, alpha=0.8, edgecolor='black',
                               linewidth=0.4, color=c,
                               label=pr.replace('Bambu_', 'Bambu '))
            lim = [0, max(x.max(), y.max()) * 1.06]
            ax.plot(lim, lim, 'r--', lw=1.1, label='y = x')
            ax.set_xlim(lim); ax.set_ylim(lim)
            ax.set_xlabel('G-code estimated print time (min)')
            ax.set_ylabel('Recording duration (min)')
            ax.set_title(f'Pearson $r$ = {r_pearson:.4f}  ($n$ = {int(ok.sum())})',
                         fontsize=11)
            ax.legend(fontsize=9, loc='lower right')
            save(fig, 'fig2_gcode_validation')
            LOG['fig2'] = {'r': round(float(r_pearson), 4), 'n': int(ok.sum())}
            print("   ", LOG['fig2'])
            made = True
if not made:
    print("   ! SKIPPED — need recording_metadata.csv and gcode_times.csv")
    print("     re-run the g-code patch cell first")

# =============================================================================
# FIG 3 — vibration feature ablation, frequency in Hz
# =============================================================================
print("\nfig3_amplitude_vs_shape")
p5 = r5.get('pooled', {})
vals = [p5.get('vib_full'), p5.get('vib_matched'),
        p5.get('amplitude'), p5.get('frequency_hz')]
labs = ['All features\n(full recording)', 'All features\n(window equalized)',
        'Amplitude\nonly', 'Frequency (Hz)\nonly']
cols = [GREY, BLUE, GREEN, PURPLE]
if all(v is not None for v in vals):
    fig, ax = plt.subplots(figsize=(6.6, 4.2))
    bars(ax, labs, vals, cols, chance=CHANCE)
    ax.legend(fontsize=9)
    save(fig, 'fig3_amplitude_vs_shape')
    LOG['fig3'] = dict(zip(labs, vals))
    print("   ", LOG['fig3'])
else:
    print("   ! SKIPPED — pooled block missing from results_v5.json")

# =============================================================================
# FIG 4 — per-printer accuracy and cross-printer transfer
# =============================================================================
print("\nfig4_per_device")
pp = r5.get('per_printer', {})
cp = r5.get('cross_printer', {})
if pp:
    names, vals, errs, cols = [], [], [], []
    for k in sorted(pp):
        d = pp[k]
        nice = k.replace('Bambu_', 'Bambu ').replace('A1mini', 'A1 Mini')
        arch = 'core-XY' if 'P1P' in k else 'bed-slinger'
        names.append(f'{nice}\n({arch})')
        vals.append(d['vib_matched'])
        lo, hi = d['ci']
        errs.append([d['vib_matched'] - lo, hi - d['vib_matched']])
        cols.append(BLUE if 'P1P' in k else ORANGE)
    for k in sorted(cp):
        d = cp[k]
        a, b_ = k.split('->')
        names.append(f"{a.replace('Bambu_','')}\n$\\rightarrow$ {b_.replace('Bambu_','')}")
        vals.append(d['acc'])
        lo, hi = d['ci']
        errs.append([d['acc'] - lo, hi - d['acc']])
        cols.append(GREY)
    errs = np.array(errs).T
    fig, ax = plt.subplots(figsize=(7.4, 4.4))
    b = ax.bar(names, vals, yerr=errs, capsize=4, color=cols,
               edgecolor='black', linewidth=0.7)
    for bb, v in zip(b, vals):
        ax.text(bb.get_x() + bb.get_width() / 2, v + 2.2, f'{v:.2f}%',
                ha='center', fontsize=9)
    ax.axhline(CHANCE, color='red', ls='--', lw=1.2, label=f'Chance ({CHANCE}%)')
    ax.set_ylabel('Vibration classification accuracy (%)')
    ax.set_ylim(0, max(vals) + 20)
    ax.legend(handles=[
        Patch(facecolor=BLUE, edgecolor='black', label='Within P1P'),
        Patch(facecolor=ORANGE, edgecolor='black', label='Within A1 Mini'),
        Patch(facecolor=GREY, edgecolor='black', label='Cross-printer transfer'),
        plt.Line2D([0], [0], color='red', ls='--', label=f'Chance ({CHANCE}%)'),
    ], fontsize=8, loc='upper right')
    ax.text(0.02, 0.96, 'Error bars: Wilson 95% CI', transform=ax.transAxes,
            fontsize=8, va='top', style='italic')
    save(fig, 'fig4_per_device')
    LOG['fig4'] = dict(zip(names, vals))
    print("   ", LOG['fig4'])
else:
    print("   ! SKIPPED — per_printer block missing from results_v5.json")

# =============================================================================
# FIG 5 — motor band vs background PSD  (recomputed from audio)
# =============================================================================
print("\nfig5_spectrum")
import librosa
from scipy.signal import welch

SKIP = {'results', 'results_v2', 'revision_v3', 'revision_v4', 'revision_v5',
        'revision_v6', 'revision_v7', 'figures', 'output', 'Printing Videos',
        'Artifacts', '__MACOSX'}
AUD = {'.mp3', '.caf', '.wav', '.m4a'}
BASE = Path(DRIVE_BASE)

print_paths, noise_paths = [], []
for p in BASE.rglob('*'):
    if not p.is_file() or p.suffix.lower() not in AUD:
        continue
    if 'Noise Recordings' in p.parts:
        noise_paths.append(p)
    elif not any(s in p.parts for s in SKIP):
        print_paths.append(p)
print(f"   print recordings: {len(print_paths)} | noise: {len(noise_paths)}")


def mean_psd(paths, n=24, offset=30.0, dur=30.0):
    acc, f = [], None
    for p in sorted(paths)[:n]:
        try:
            y, sr = librosa.load(str(p), sr=16000, mono=True,
                                 offset=offset, duration=dur)
        except Exception:
            continue
        if len(y) < 8192:
            continue
        f, P = welch(y, fs=sr, nperseg=4096)
        acc.append(P)
    return (f, np.mean(acc, axis=0)) if acc else (None, None)


f_pr, P_pr = mean_psd(print_paths, n=24, offset=30.0)
f_no, P_no = mean_psd(noise_paths, n=24, offset=0.0)

if P_pr is not None:
    band = (f_pr >= 120) & (f_pr <= 360)
    ref = (f_pr >= 600) & (f_pr <= 2000)
    ratio_pr = 10 * np.log10(P_pr[band].mean() / P_pr[ref].mean())
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    ax.axvspan(120, 360, color=ORANGE, alpha=0.22, zorder=0,
               label='Motor resonance band (120--360 Hz)')
    ax.semilogy(f_pr, P_pr, lw=1.2, color=BLUE, label='During printing')
    if P_no is not None:
        ratio_no = 10 * np.log10(P_no[band].mean() / P_no[ref].mean())
        diff = ratio_pr - ratio_no
        ax.semilogy(f_no, P_no, lw=1.2, color=GREY, label='Background noise')
        ax.set_title(f'Motor band elevated {diff:+.2f} dB during printing',
                     fontsize=11)
        LOG['fig5'] = {'print_dB': round(float(ratio_pr), 2),
                       'noise_dB': round(float(ratio_no), 2),
                       'difference_dB': round(float(diff), 2)}
    else:
        LOG['fig5'] = {'print_dB': round(float(ratio_pr), 2)}
        print("   ! no background recordings found — single-trace figure")
    ax.set_xlim(0, 3000)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Power spectral density')
    ax.legend(fontsize=9)
    save(fig, 'fig5_spectrum')
    print("   ", LOG['fig5'])
else:
    print("   ! SKIPPED — could not load audio")

# =============================================================================
print("\n" + "=" * 70)
print("FIGURE VALUES — cross-check these against the reported results")
print("=" * 70)
print(json.dumps(LOG, indent=2, default=str))
Path(f'{OUT}/figure_values.json').write_text(json.dumps(LOG, indent=2, default=str))

made = sorted(Path(OUT).glob('fig*.png'))
print(f"\n{len(made)} figures in {OUT}:")
for m in made:
    print("   ", m.name)
expected = {'fig1_acoustic_window.png', 'fig2_gcode_validation.png',
            'fig3_amplitude_vs_shape.png', 'fig4_per_device.png',
            'fig5_spectrum.png'}
missing = expected - {m.name for m in made}
extra = {m.name for m in made} - expected
if missing:
    print("\n!! MISSING:", sorted(missing))
if extra:
    print("\n!! RETIRED FIGURES PRESENT — delete before pushing:", sorted(extra))
if not missing and not extra:
    print("\nAll five summary figures present, nothing retired left behind.")

Mounted at /content/drive

fig1_acoustic_window
  saved fig1_acoustic_window
    {'10 s': 6.94, '30 s': 11.11, '60 s': 27.08, '120 s': 30.56, '300 s': 31.25, '8 x 30 s\ndistributed': 40.28}

fig2_gcode_validation
  saved fig2_gcode_validation
    {'r': 0.9073, 'n': 144}

fig3_amplitude_vs_shape
  saved fig3_amplitude_vs_shape
    {'All features\n(full recording)': 29.17, 'All features\n(window equalized)': 22.22, 'Amplitude\nonly': 26.39, 'Frequency (Hz)\nonly': 9.03}

fig4_per_device
  saved fig4_per_device
    {'Bambu A1 Mini\n(bed-slinger)': 13.89, 'Bambu P1P\n(core-XY)': 36.11, 'A1mini\n$\\rightarrow$ P1P': 8.33, 'P1P\n$\\rightarrow$ A1mini': 13.89}

fig5_spectrum
   print recordings: 144 | noise: 6


/tmp/ipykernel_4932/3340714495.py:294: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(str(p), sr=16000, mono=True,
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_4932/3340714495.py:294: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(str(p), sr=16000, mono=True,
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_4932/3340714495.py:294: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(str(p), sr=16000, mono=True,
/usr/local/l

  saved fig5_spectrum
    {'print_dB': 9.25, 'noise_dB': 4.33, 'difference_dB': 4.92}

FIGURE VALUES — cross-check these against the reported results
{
  "fig1": {
    "10 s": 6.94,
    "30 s": 11.11,
    "60 s": 27.08,
    "120 s": 30.56,
    "300 s": 31.25,
    "8 x 30 s\ndistributed": 40.28
  },
  "fig2": {
    "r": 0.9073,
    "n": 144
  },
  "fig3": {
    "All features\n(full recording)": 29.17,
    "All features\n(window equalized)": 22.22,
    "Amplitude\nonly": 26.39,
    "Frequency (Hz)\nonly": 9.03
  },
  "fig4": {
    "Bambu A1 Mini\n(bed-slinger)": 13.89,
    "Bambu P1P\n(core-XY)": 36.11,
    "A1mini\n$\\rightarrow$ P1P": 8.33,
    "P1P\n$\\rightarrow$ A1mini": 13.89
  },
  "fig5": {
    "print_dB": 9.25,
    "noise_dB": 4.33,
    "difference_dB": 4.92
  }
}

5 figures in /content/drive/MyDrive/3dprinter_sidechannel/figures:
    fig1_acoustic_window.png
    fig2_gcode_validation.png
    fig3_amplitude_vs_shape.png
    fig4_per_device.png
    fig5_spectrum.png

All five sum